# LIANA (ileum) — epithelial–glia & FARM–T interactions

Ileum LIANA results (`ileum_corrected_liana.csv`) are large (~40M rows). The notebook reads the CSV **once** in chunks, keeps only rows touching predefined `source`/`target` labels, then subsets for neurotransmitter / axon-related ligand–receptor names.

Atlas cell types in `AnnData` use `obs['hgca_celltype_v1']` (loaded in **backed** mode for metadata only).

### Figures
Exports follow the [Nature research figure guide](https://research-figure-guide.nature.com/figures/): **Helvetica**, **black** labels, colour-blind–friendly palette from *Building and exporting figure panels*, **vector PDF** (`pdf.fonttype=42`). Further Illustrator edits → save as **.ai**; otherwise use the **.pdf** from this notebook.
### Neuropod / NCAM / GDNF–RET module
§**2.2** adds **Glia ↔ EEC I (CCK+)** to the epithelial–glia watchlist, then filters ileum LIANA for **NCAM/IgCAM**, **GDNF-family ligands and RET/GFRα co-receptors**, and **neurofilament** L–R tokens, plus optional **NEFL/NEFM/NEFH/NCAM1/GDNF/RET/GFRA1** expression summaries. **Re-run** the chunked LIANA load cell after pulling this update so `EEC I Cells (CCK+)` rows enter `dfw`.


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# --- paths (edit if your GCA tree differs) ---
GCA_ROOT = Path.home() / "Projects" / "GCA"
INTEGRATED = GCA_ROOT / "meta_datasets" / "integrated-objects" / "hgca_all_lineages_v1.h5ad"
LIANA_ILEUM = GCA_ROOT / "downstream_data" / "ileum_corrected_liana.csv"
FIG_DIR = GCA_ROOT / "github_vignette_output" / "LIANA" / "ileum_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("INTEGRATED:", INTEGRATED, "exists:", INTEGRATED.is_file())
print("LIANA_ILEUM:", LIANA_ILEUM, "exists:", LIANA_ILEUM.is_file())
print("FIG_DIR:", FIG_DIR)

INTEGRATED: /Users/kylekimler/Projects/GCA/meta_datasets/integrated-objects/hgca_all_lineages_v1.h5ad exists: True
LIANA_ILEUM: /Users/kylekimler/Projects/GCA/downstream_data/ileum_corrected_liana.csv exists: True
FIG_DIR: /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures


/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
# Nature — "Building and exporting figure panels" palette (colour-blind friendly)
# https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/
NATURE_WONG = {
    "black": "#000000",
    "orange": "#e69f00",
    "sky_blue": "#56b4e9",
    "bluish_green": "#009e73",
    "yellow": "#f0e442",
    "blue": "#0072b2",
    "vermillion": "#d55e00",
    "reddish_purple": "#cc79a7",
}
NATURE_CYCLE = [
    NATURE_WONG["orange"],
    NATURE_WONG["sky_blue"],
    NATURE_WONG["bluish_green"],
    NATURE_WONG["blue"],
    NATURE_WONG["vermillion"],
    NATURE_WONG["reddish_purple"],
    NATURE_WONG["yellow"],
]


def setup_nature_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Helvetica", "Helvetica Neue", "Arial", "DejaVu Sans"],
            "font.size": 7,
            "axes.labelsize": 7,
            "axes.titlesize": 7,
            "figure.titlesize": 7,
            "xtick.labelsize": 6,
            "ytick.labelsize": 6,
            "legend.fontsize": 6,
            "axes.linewidth": 0.6,
            "xtick.major.width": 0.6,
            "ytick.major.width": 0.6,
            "lines.linewidth": 0.9,
            "axes.grid": False,
            "text.color": NATURE_WONG["black"],
            "axes.labelcolor": NATURE_WONG["black"],
            "axes.edgecolor": NATURE_WONG["black"],
            "xtick.color": NATURE_WONG["black"],
            "ytick.color": NATURE_WONG["black"],
            "axes.titlecolor": NATURE_WONG["black"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.transparent": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
        }
    )


def save_pdf(fig: mpl.figure.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        path,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.08,
        metadata={"Creator": "LIANA_analysis.ipynb"},
    )
    plt.close(fig)
    print("Saved", path)


def shorten_label(s: str, max_len: int = 46) -> str:
    s = str(s)
    return s if len(s) <= max_len else s[: max_len - 1] + "…"


def add_lr_pair_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["lr_pair"] = (
        out["ligand_complex"].astype(str) + " — " + out["receptor_complex"].astype(str)
    ).map(shorten_label)
    return out


def max_lr_per_pair(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["ligand_complex", "receptor_complex", "lrscore"])
    return (
        df.groupby(["ligand_complex", "receptor_complex"], as_index=False)["lrscore"]
        .max()
        .sort_values("lrscore", ascending=False)
    )


def panel_label(ax: mpl.axes.Axes, tag: str) -> None:
    ax.text(
        0.0,
        1.02,
        tag,
        transform=ax.transAxes,
        fontsize=8,
        fontweight="bold",
        color=NATURE_WONG["black"],
        va="bottom",
        ha="left",
    )


def hbar_top_lr(df: pd.DataFrame, ax: mpl.axes.Axes, title: str, n: int = 10) -> None:
    d = add_lr_pair_column(df.head(n))
    if d.empty:
        ax.set_axis_off()
        ax.text(0.5, 0.5, "No interactions", ha="center", va="center", fontsize=7, color=NATURE_WONG["black"])
        return
    colors = [NATURE_CYCLE[i % len(NATURE_CYCLE)] for i in range(len(d))]
    y = np.arange(len(d))
    ax.barh(y, d["lrscore"], color=colors, edgecolor=NATURE_WONG["black"], linewidth=0.35)
    ax.set_yticks(y)
    ax.set_yticklabels(d["lr_pair"], fontsize=5.5, color=NATURE_WONG["black"])
    ax.invert_yaxis()
    ax.set_xlabel("LIANA lrscore (unitless)", color=NATURE_WONG["black"])
    ax.set_title(title, fontsize=7, color=NATURE_WONG["black"])
    ax.tick_params(axis="both", colors=NATURE_WONG["black"])
    ax.tick_params(axis="x", which="major", length=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


NATURE_CMAP = LinearSegmentedColormap.from_list(
    "nat_seq", ["#ffffff", NATURE_WONG["sky_blue"], NATURE_WONG["blue"], NATURE_WONG["black"]]
)

# LIANA direction (same convention for all figures: ligand on source, receptor on target)
LIANA_NOTE_GLIA_HEATMAP = (
    "Direction: ligand on source cells, receptor on target cells (LIANA source/target). "
    "Row label = ligand and receptor (not an arrow). "
    "Each cell = max lrscore over Glia-to-epithelium and epithelium-to-Glia for that L-R pair."
)
LIANA_NOTE_GLIA_BARS = (
    "Direction: ligand on source, receptor on target; each bar = max lrscore over both directions "
    "(Glia-to-epithelium and epithelium-to-Glia)."
)
LIANA_NOTE_FARM_BARS = (
    "Direction: ligand on source, receptor on target; each bar = max lrscore over both directions "
    "(FARM-to-CD4 and CD4-to-FARM)."
)
LIANA_NOTE_FARM_HEATMAP = (
    "Direction: ligand on source, receptor on target; each cell = max lrscore over FARM-to-CD4 and CD4-to-FARM."
)
LIANA_NOTE_SHORT = (
    "Direction: ligand on source, receptor on target; each plotted value = max lrscore over both directions (A-to-B and B-to-A)."
)

setup_nature_style()

## 1. Atlas metadata (`hgca_celltype_v1`)
Backed read: **no** expression matrix in memory.

In [3]:
adata = sc.read_h5ad(INTEGRATED, backed="r")
assert "hgca_celltype_v1" in adata.obs.columns, adata.obs.columns[:20]

ct = adata.obs["hgca_celltype_v1"].astype("string")
print(adata.n_obs, "cells")
display(ct.value_counts().head(25))

for label in [
    "Glia",
    "EEC L",
    "EEC Progenitors",
    "BEST4 Enterocytes",
    "Tuft Cells",
    "Follicle Associated Resident Macrophages",
    "CD4 Tr1",
    "CD4 Tfr",
    "Gamma Delta T Cells",
    "NKT Cells",
]:
    n = int((ct == label).sum())
    print(f"{label!r}: {n}")

if hasattr(adata, "file") and adata.file is not None:
    adata.file.close()

944502 cells


hgca_celltype_v1
CD8 TRM                              130081
Villus Tip Enterocytes                62858
CD4 Memory                            55896
Memory B                              45602
CD4 Th17                              45218
CD8 IEL                               41143
Plasma Cells                          38083
Naive B                               33898
Plasma IGA                            31875
Mid Villus Enterocytes                30619
CD4 Tfh                               24832
Lower Crypt Colonocytes               21199
Goblet Cells                          20309
CD8 Circulating Effector Memory       18471
Transiently Amplifying Cells (TA)     18299
Lower Villus Enterocytes              18149
Crypt Top Colonocytes                 18075
Mast Cells                            17702
Mature Goblet Cells                   17419
NKT Cells                             17114
CD4 Naive                             16416
Lamina propria Fibroblasts (S1)       16327
Colonocyte Prog

'Glia': 2604
'EEC L': 473
'EEC Progenitors': 546
'BEST4 Enterocytes': 2691
'Tuft Cells': 2993
'Follicle Associated Resident Macrophages': 902
'CD4 Tr1': 1878
'CD4 Tfr': 2407
'Gamma Delta T Cells': 2946
'NKT Cells': 17114


## 2. Load ileum LIANA efficiently (one chunked pass)

LIANA column names follow the export: `source`, `target`, `ligand_complex`, `receptor_complex`, `lrscore`, …

We first restrict to rows where **either** `source` or `target` is in a small whitelist of cell-type strings used below, then assign each row to interaction groups.

In [4]:
# LIANA label sets (must match strings in the CSV)
GLIA = {"Glia"}

EEC_L = {"EEC L"}
# I cells (CCK+); label must match hgca_celltype_v1 in the LIANA CSV
EEC_I = {"EEC I Cells (CCK+)", "EEC I"}  # second alias if present in hgca_celltype_v1
# Mature EEC (exclude progenitors; those are analysed separately)
EEC_MAT = {
    "EEC N",
    "EEC S",
    "EEC L",
    "EEC I Cells (CCK+)",
    "EEC I",
    "Enteroendocrine Cells (EEC)",
    "EEC Enterochromaffin (EC)",
}
BEST4_ILEUM = {"BEST4 Enterocytes"}  # ileum BEST4 enterocytes
BEST4_COLON = {"BEST4 Colonocytes"}
BEST4_BOTH = {"BEST4 Enterocytes", "BEST4 Colonocytes"}
VILLUS_BOTTOM = {"Lower Villus Enterocytes"}  # control epithelium (non-neuropod-focused)
TUFT = {"Tuft Cells"}
TUFT_PROG = {"Tuft Progenitors"}
EEC_PROG = {"EEC Progenitors"}
SEC_PROG = {"Secretory Progenitors"}

FARM = {"Follicle Associated Resident Macrophages"}
T_WITH_FARM = {
    "CD4 Tr1",
    "CD4 Tfr",
    "CD4 Tfh",
    "CD4 pTreg",
    "CD4 Th17",
    "CD4 tTreg",
    "NKT Cells",
    "Gamma Delta T Cells",
}

ALL_WATCH = (
    GLIA
    | EEC_MAT
    | BEST4_BOTH
    | VILLUS_BOTTOM
    | TUFT
    | TUFT_PROG
    | EEC_PROG
    | SEC_PROG
    | FARM
    | T_WITH_FARM
)


def bidirectional_mask(df: pd.DataFrame, a: set[str], b: set[str]) -> pd.Series:
    return (
        (df["source"].isin(a) & df["target"].isin(b))
        | (df["source"].isin(b) & df["target"].isin(a))
    )


def neuro_axon_mask(df: pd.DataFrame) -> pd.Series:
    """Heuristic keyword filter on ligand/receptor complex symbols."""
    lr = (
        df["ligand_complex"].astype(str)
        + "|"
        + df["receptor_complex"].astype(str)
    ).str.upper()
    neuro = [
        "GABA",
        "GABR",
        "GRIA",
        "GRIN",
        "GRM",
        "GAD",
        "SLC6A",
        "SLC17A",
        "SLC32A",
        "HTR",
        "DRD",
        "CHRM",
        "CHRNA",
        "CHRNB",
        "TPH",
        "DDC",
        "DBH",
        "COMT",
        "MAOA",
        "ACHE",
        "CHAT",
        "SLC18A",
        "ADRA",
        "ADRB",
        "DOP",
        "5-HT",
        "SEROT",
        "GLUT",
        "NOS1",
        "NTS",
        "TACR",
        "GRP",
        "CCK",
        "PYY",
        "NPY",
        "POMC",
    ]
    axon = [
        "NGF",
        "NTRK",
        "TRK",
        "BDNF",
        "GDNF",
        "ROBO",
        "SLIT",
        "SEMA",
        "PLXA",
        "NRP",
        "UNC5",
        "DCC",
        "RTN",
        "L1CAM",
        "NCAM",
        "NRXN",
        "NLGN",
        "CNTN",
        "LINGO",
        "EPHA",
        "EPHB",
        "EFNA",
        "EFNB",
        "WNT",
        "NTN",
        "PTPR",
        "PTN",
    ]
    keys = sorted(set(neuro + axon))
    pat = "|".join(re.escape(k) for k in keys)
    return lr.str.contains(pat, regex=True, na=False)


def _primary_ligand_token(s: str) -> str:
    """First symbol of LIANA ligand_complex (handles PYY^DPP4-style complexes)."""
    t = str(s).strip().upper()
    for sep in ("^", "+", "_", "|"):
        if sep in t:
            t = t.split(sep, 1)[0]
            break
    return t.strip()


def reduce_canonical_neuropeptide_outgoing(df: pd.DataFrame) -> pd.DataFrame:
    """Keep one canonical outgoing L–R per PYY, SST, NTS, TPH1 (best lrscore among canonical receptors). Drops non-canonical rows; if no canonical receptor, drops all rows for that hormone."""
    if df.empty:
        return df
    out = df.copy()
    lig_tok = out["ligand_complex"].map(_primary_ligand_token)
    rec_u = out["receptor_complex"].astype(str).str.upper()
    # Canonical receptor priority (gut / neuroendocrine literature)
    CANONICAL = {
        "PYY": ["NPY1R", "NPY2R"],
        "SST": ["SSTR2", "SSTR1", "SSTR5"],
        "NTS": ["NTSR1", "NTSR2", "NGFR", "SORT1"],
        "TPH1": ["HTR1A", "HTR2A", "HTR3A", "HTR1B"],
    }
    drop_idx: list = []
    for horm, recs in CANONICAL.items():
        m = lig_tok == horm.upper()
        if not m.any():
            continue
        idx_all = out.index[m]
        best_idx = None
        best_score = -np.inf
        for r in recs:
            mm = m & (rec_u == r.upper())
            if mm.any():
                bi = out.loc[mm, "lrscore"].idxmax()
                sc = float(out.loc[bi, "lrscore"])
                if sc > best_score:
                    best_score = sc
                    best_idx = bi
        if best_idx is None:
            drop_idx.extend(idx_all.tolist())
        else:
            drop_idx.extend([i for i in idx_all if i != best_idx])
    return out.drop(index=drop_idx, errors="ignore")


USECOLS = [
    "source",
    "target",
    "ligand_complex",
    "receptor_complex",
    "lr_means",
    "lrscore",
    "cellphone_pvals",
    "spec_weight",
]


def load_liana_watchlist(
    path: Path,
    watch: set[str],
    chunksize: int = 1_000_000,
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for i, chunk in enumerate(pd.read_csv(path, usecols=USECOLS, chunksize=chunksize)):
        sub = chunk[chunk["source"].isin(watch) | chunk["target"].isin(watch)]
        if len(sub):
            parts.append(sub)
        if (i + 1) % 10 == 0:
            print(f"chunks processed: {i + 1}, rows kept so far: {sum(len(p) for p in parts):,}")
    if not parts:
        return pd.DataFrame(columns=USECOLS)
    return pd.concat(parts, ignore_index=True)


dfw = load_liana_watchlist(LIANA_ILEUM, ALL_WATCH)
print("Filtered rows (touching watchlist cell types):", len(dfw))
dfw.head()

chunks processed: 10, rows kept so far: 4,003,403
chunks processed: 20, rows kept so far: 8,208,782
chunks processed: 30, rows kept so far: 12,279,288
Filtered rows (touching watchlist cell types): 16196312


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
0,Mature Goblet Cells,Gamma Delta T Cells,TFF3,CXCR4,1580.0220,0.0,0.015594,0.993948
1,Neutrophils,CD4 Th17,S100A8,CD69,1532.0663,0.0,0.045527,0.993989
2,Mature Goblet Cells,CD4 Tfh,TFF3,CXCR4,1554.8007,0.0,0.011118,0.992840
3,Mature Goblet Cells,CD4 Tfr,TFF3,CXCR4,1544.6865,0.0,0.009324,0.992187
4,Mature Goblet Cells,CD4 Th17,TFF3,CXCR4,1542.5238,0.0,0.008940,0.992022


### 2.1 Epithelium / progenitors ↔ Glia
Bidirectional; sorted by `lrscore`.

In [5]:
groups = {
    "EEC L <-> Glia": bidirectional_mask(dfw, EEC_L, GLIA),
    "EEC I (CCK+) <-> Glia": bidirectional_mask(dfw, EEC_I, GLIA),
    "EEC (mature) <-> Glia": bidirectional_mask(dfw, EEC_MAT, GLIA),
    "BEST4 Enterocyte <-> Glia": bidirectional_mask(dfw, BEST4_ILEUM, GLIA),
    "BEST4 Colonocytes <-> Glia": bidirectional_mask(dfw, BEST4_COLON, GLIA),
    "Tuft <-> Glia": bidirectional_mask(dfw, TUFT, GLIA),
    "EEC Progenitors <-> Glia": bidirectional_mask(dfw, EEC_PROG, GLIA),
    "Tuft Progenitors <-> Glia": bidirectional_mask(dfw, TUFT_PROG, GLIA),
    "Secretory Progenitors <-> Glia": bidirectional_mask(dfw, SEC_PROG, GLIA),
    "Villus-bottom enterocytes <-> Glia": bidirectional_mask(dfw, VILLUS_BOTTOM, GLIA),
}

glia_tables = {}
for name, m in groups.items():
    sub = dfw.loc[m].copy()
    sub = sub.sort_values("lrscore", ascending=False)
    sub = reduce_canonical_neuropeptide_outgoing(sub)
    glia_tables[name] = sub
    print(f"\n=== {name} ===")
    print("n interactions:", len(sub))
    if len(sub):
        display(sub.head(15))
        na = sub[neuro_axon_mask(sub)]
        print("  neuro/axon keyword hits:", len(na))
        if len(na):
            display(na.head(20))


=== EEC L <-> Glia ===
n interactions: 8916


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
2726,EEC L,Glia,NTS,NGFR,353.027830,0.000,0.084818,0.975630
9838,Glia,EEC L,LGALS1,ITGB1,175.177100,0.000,0.000536,0.953212
4324,Glia,EEC L,VIM,CD44,280.384980,0.000,0.000256,0.949593
54492,EEC L,Glia,MDK,PTPRZ1,69.008590,0.000,0.060507,0.937370
51797,EEC L,Glia,HLA-A,APLP2,70.918400,1.000,0.000168,0.934046
34982,Glia,EEC L,S100A4,ERBB3,88.815450,0.000,0.001447,0.930978
19581,Glia,EEC L,SPP1,CD44,122.733850,0.000,0.004539,0.924691
100154,EEC L,Glia,TTR,NGFR,49.391304,0.000,0.114665,0.917962
91663,Glia,EEC L,TIMP1,CD63,52.005157,0.000,0.000118,0.916572
33060,Glia,EEC L,HLA-A,APLP2,91.757950,0.001,0.000097,0.915079


  neuro/axon keyword hits: 2125


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
2726,EEC L,Glia,NTS,NGFR,353.027830,0.000,0.084818,0.975630
54492,EEC L,Glia,MDK,PTPRZ1,69.008590,0.000,0.060507,0.937370
100154,EEC L,Glia,TTR,NGFR,49.391304,0.000,0.114665,0.917962
79321,Glia,EEC L,L1CAM,ERBB3,56.380653,0.000,0.040639,0.911346
149730,EEC L,Glia,RTN4,NGFR,38.121513,0.000,0.008577,0.893486
94049,Glia,EEC L,NCAM1,CACNA1C,51.224140,0.000,0.018512,0.892414
97740,EEC L,Glia,CALM1,PTPRA,50.344807,0.000,0.000270,0.888977
96169,Glia,EEC L,L1CAM,CD9,50.725388,0.000,0.004383,0.879625
176863,EEC L,Glia,APP,NGFR,34.357460,0.000,0.009033,0.878264
101761,Glia,EEC L,NCAM1,PTPRA,48.849040,0.000,0.004741,0.873333



=== EEC (mature) <-> Glia ===
n interactions: 44564


...

,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
63877,Glia,Secretory Progenitors,L1CAM,CD9,63.298590,0.0,0.013922,0.928694
138877,Secretory Progenitors,Glia,MDK,PTPRZ1,40.170876,0.0,0.021359,0.898911
95615,Glia,Secretory Progenitors,L1CAM,EPHB2,50.831860,0.0,0.042574,0.880588
97560,Glia,Secretory Progenitors,L1CAM,ERBB3,50.407745,0.0,0.019405,0.876598
192583,Secretory Progenitors,Glia,APP,NGFR,32.569927,0.0,0.007523,0.868146
193047,Secretory Progenitors,Glia,RTN4,NGFR,32.526947,0.0,0.005259,0.867869
167498,Secretory Progenitors,Glia,CALM1,PTPRA,35.626700,1.0,0.000178,0.866636
366290,Glia,Secretory Progenitors,SLIT2,APP,21.268684,0.0,0.001694,0.826350
107718,Glia,Secretory Progenitors,L1CAM,ERBB2,47.253517,0.0,0.019740,0.821922
108245,Glia,Secretory Progenitors,L1CAM,EGFR,47.103233,0.0,0.013015,0.816933



=== Villus-bottom enterocytes <-> Glia ===
n interactions: 8916


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
10237,Glia,Lower Villus Enterocytes,LGALS1,ITGB1,170.820420,0.0,0.000348,0.942554
20931,Lower Villus Enterocytes,Glia,LGALS3,MCAM,119.828500,0.0,0.002578,0.942262
69644,Lower Villus Enterocytes,Glia,HLA-A,APLP2,60.391937,1.0,0.000134,0.926770
32020,Glia,Lower Villus Enterocytes,HLA-A,APLP2,93.957140,0.0,0.000130,0.925565
72061,Glia,Lower Villus Enterocytes,L1CAM,CD9,59.434845,0.0,0.010990,0.920458
88871,Glia,Lower Villus Enterocytes,APP,CD74,53.013763,1.0,0.000073,0.919432
37178,Glia,Lower Villus Enterocytes,S100A4,ERBB3,85.511520,0.0,0.001029,0.919180
22933,Glia,Lower Villus Enterocytes,S100A10,CFTR,115.045500,0.0,0.001237,0.916423
4460,Glia,Lower Villus Enterocytes,VIM,CD44,276.150820,0.0,0.000084,0.915114
63134,Glia,Lower Villus Enterocytes,LAMB2,RPSA,63.793526,0.0,0.000910,0.909093


  neuro/axon keyword hits: 2125


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
72061,Glia,Lower Villus Enterocytes,L1CAM,CD9,59.434845,0.0,0.010990,0.920458
72623,Lower Villus Enterocytes,Glia,CALM1,PTPRA,59.165230,0.0,0.000325,0.897836
88698,Glia,Lower Villus Enterocytes,L1CAM,ERBB3,53.076720,0.0,0.028893,0.896566
158214,Lower Villus Enterocytes,Glia,RTN4,NGFR,36.780148,0.0,0.007782,0.888764
203329,Lower Villus Enterocytes,Glia,APP,NGFR,31.512733,0.0,0.006630,0.860746
101333,Glia,Lower Villus Enterocytes,L1CAM,ERBB2,49.009052,0.0,0.034778,0.859675
112877,Glia,Lower Villus Enterocytes,NCAM1,PTPRA,45.801647,0.0,0.002020,0.818179
397836,Glia,Lower Villus Enterocytes,SLIT2,APP,20.211490,0.0,0.001493,0.817099
108549,Glia,Lower Villus Enterocytes,L1CAM,EPHB2,47.010246,0.0,0.014917,0.813611
109508,Glia,Lower Villus Enterocytes,L1CAM,EGFR,46.723630,0.0,0.010722,0.801991


### 2.2 Glia ↔ EEC neuropod focus: NCAM, GDNF / RET / GFRα L–R, neurofilament

**Context:** Recent 3D microscopy / ultrastructure work reports direct glial contacts on EEC neuropods (including **NCAM**-related adhesion). **GLP-1 / EEC L** populations are enriched **distally** (ileum, some colon); **CCK / EEC I** may enrich more **proximally**. The precomputed table here is **`ileum_corrected_liana.csv`** (ileum-centric CCC); use it to see if overlapping **L–R** signal appears for **Glia ↔ EEC L** vs **Glia ↔ EEC I (CCK+)**. Colon- or duodenum-specific CCC would need separate LIANA outputs.

This block:
1. **Filters** Glia↔EEC L and Glia↔EEC I tables for curated **token sets** (NCAM / IgCAM; **GDNF-family ligands** (**GDNF**, **NRTN**, **ARTN**, **PSPN**) plus **RET** and **GFRA1–4** co-receptors; **neurofilament** genes if they appear as L–R in consensus).
2. Writes **`neuropod_glia_eec_focus_summary.csv`** (top hits per panel) and bar PDFs under `FIG_DIR`.
3. Optionally **mean `log1p` expression** of **NEFL / NEFM / NEFH / NCAM1 / GDNF / RET / GFRA1** in EEC subsets vs other epithelial types (ileum vs duodenum/jejunum vs colon from `tissue_ontology_term`), using the integrated object in **backed** mode (slow on first run).


In [ ]:
# --- Curated tokens: NCAM/neuropod adhesion; GDNF-family + RET/GFRa; neurofilament ---
# (Substring match on ligand_complex | receptor_complex, uppercase; extend as needed.)
NEUROPOD_NCAM_IGCAM_TOKENS = {
    "NCAM1", "NCAM2", "NCAML1", "L1CAM", "CHL1", "DSCAM", "OPCML", "PTPRF",
    "CADM1", "CADM2", "CADM3", "NRXN", "NLGN", "CNTN", "LRRTM", "LRRC4",
}
NEUROPOD_GDNF_RET_GFRA_TOKENS = {
    "GDNF",
    "NRTN",
    "ARTN",
    "PSPN",
    "RET",
    "GFRA1",
    "GFRA2",
    "GFRA3",
    "GFRA4",
}
NEUROPOD_NEUROFIL_TOKENS = {"NEFL", "NEFM", "NEFH", "PRPH", "INA"}


def _lr_upper_concat(df: pd.DataFrame) -> pd.Series:
    return (
        df["ligand_complex"].astype(str) + "|" + df["receptor_complex"].astype(str)
    ).str.upper()


def mask_lr_token_hits(df: pd.DataFrame, tokens: set[str]) -> pd.Series:
    lr = _lr_upper_concat(df)
    pat = "|".join(re.escape(t) for t in sorted(tokens))
    return lr.str.contains(pat, regex=True, na=False)


def filter_glia_eec_neuropod_focus(
    table: pd.DataFrame,
    *,
    name: str,
    token_set: set[str],
    set_label: str,
) -> pd.DataFrame:
    if table is None or table.empty:
        return pd.DataFrame()
    m = bidirectional_mask(table, EEC_L | EEC_I | EEC_MAT, GLIA)
    sub = table.loc[m].copy()
    sub = sub[mask_lr_token_hits(sub, token_set)]
    if sub.empty:
        return sub
    sub = sub.assign(context=name, gene_set=set_label)
    return sub.sort_values("lrscore", ascending=False)


def export_neuropod_focus_tables() -> pd.DataFrame:
    rows_out = []
    eec_keys = ("EEC L <-> Glia", "EEC I (CCK+) <-> Glia")
    focus_spec = (
        ("NCAM_IgCAM", NEUROPOD_NCAM_IGCAM_TOKENS),
        ("GDNF_RET_GFRA", NEUROPOD_GDNF_RET_GFRA_TOKENS),
        ("neurofilament", NEUROPOD_NEUROFIL_TOKENS),
    )
    for gkey in eec_keys:
        base = glia_tables.get(gkey)
        if base is None or base.empty:
            print(f"  skip focus: empty glia_tables['{gkey}']")
            continue
        for tag, tok in focus_spec:
            hit = filter_glia_eec_neuropod_focus(base, name=gkey, token_set=tok, set_label=tag)
            if hit.empty:
                print(f"  no hits: {gkey} / {tag}")
                continue
            top = hit.head(25).copy()
            rows_out.append(top)
            fig, ax = plt.subplots(figsize=(5.5, 4.2))
            hbar_top_lr(top, ax, title=f"{gkey}: {tag} (top 10)", n=10)
            panel_label(ax, "")
            fig.tight_layout(pad=0.6)
            safe = gkey.replace(" ", "_").replace("(", "").replace(")", "").replace("+", "p")
            save_pdf(fig, FIG_DIR / f"fig_neuropod_glia_{safe}_{tag}.pdf")
    if not rows_out:
        return pd.DataFrame()
    comb = pd.concat(rows_out, ignore_index=True)
    outp = FIG_DIR / "neuropod_glia_eec_focus_summary.csv"
    outp.parent.mkdir(parents=True, exist_ok=True)
    comb.to_csv(outp, index=False)
    print("Wrote", outp, "rows", len(comb))
    return comb


_neu_df = export_neuropod_focus_tables()
if _neu_df is not None and len(_neu_df):
    display(_neu_df.head(40))


# --- Optional: neurofilament + NCAM1 mean log1p by EEC type and broad tissue region ---
def _tissue_bucket(s: str) -> str:
    t = str(s).lower()
    if "ileum" in t:
        return "ileum"
    if "colon" in t:
        return "colon"
    if any(x in t for x in ("duoden", "jejun", "stomach", "fundus", "pylor")):
        return "proximal_gut"
    return "other"


def plot_neurofil_ncam_expr_by_eec(
    genes: tuple[str, ...] = ("NEFL", "NEFM", "NEFH", "NCAM1", "GDNF", "RET", "GFRA1"),
    max_cells_per_stratum: int = 4000,
) -> None:
    if not INTEGRATED.is_file():
        print("INTEGRATED h5ad missing; skip expression panel")
        return
    try:
        ad = sc.read_h5ad(INTEGRATED, backed="r")
    except Exception as e:
        print("Could not open INTEGRATED:", e)
        return
    obs = ad.obs
    ct = obs["hgca_celltype_v1"].astype(str)
    tissue_col = "tissue_ontology_term" if "tissue_ontology_term" in obs.columns else None
    eec_types = sorted(set(EEC_MAT) | EEC_L | EEC_I)
    mask_epi = ct.isin(eec_types) | (
        ct.str.contains("Enterocyte", case=False, na=False)
        & ~ct.str.contains("Progenitor", case=False, na=False)
    )
    mask_glia = ct.eq("Glia")
    mask_plot = mask_epi | mask_glia
    if not mask_plot.any():
        print("No EEC/enterocyte/glia cells found for expression panel")
        return
    # gene names: prefer var['gene_symbol'] if present
    gsym = None
    if "gene_symbol" in ad.var.columns:
        gsym = ad.var["gene_symbol"].astype(str).values
    var_names = np.array(ad.var_names.astype(str))
    def _gene_idx(g: str):
        if gsym is not None:
            jj = np.flatnonzero(gsym == g)
            if len(jj):
                return int(jj[0])
        jj = np.flatnonzero(var_names == g)
        return int(jj[0]) if len(jj) else None

    idxs = {g: _gene_idx(g) for g in genes}
    missing = [g for g, j in idxs.items() if j is None]
    if missing:
        print("Missing genes in object (skipped):", missing)
    idxs = {g: j for g, j in idxs.items() if j is not None}
    if not idxs:
        return

    sub_idx = np.flatnonzero(mask_plot.to_numpy())
    if len(sub_idx) > max_cells_per_stratum:
        rng = np.random.default_rng(0)
        sub_idx = rng.choice(sub_idx, size=max_cells_per_stratum, replace=False)
        sub_idx = np.sort(sub_idx)

    want_names = [ad.var_names[j] for j in idxs.values()]
    Xchunk = ad[sub_idx, want_names].X
    if hasattr(Xchunk, "toarray"):
        Xchunk = Xchunk.toarray()
    mat = np.asarray(Xchunk, dtype=float)
    sub_obs = obs.iloc[sub_idx].copy()
    sub_obs = sub_obs.assign(
        broad_region=sub_obs[tissue_col].map(_tissue_bucket) if tissue_col else "all",
        cell_group=ct.iloc[sub_idx].astype(str),
    )
    for i, (gn, col_j) in enumerate(zip(idxs.keys(), range(mat.shape[1]))):
        sub_obs[gn] = np.log1p(mat[:, col_j])

    # Aggregate: EEC I vs EEC L vs Glia vs enterocyte (non-EEC)
    rows = []
    for (region, cgroup), gg in sub_obs.groupby(["broad_region", "cell_group"], sort=False):
        chunk = {g: float(gg[g].mean()) for g in idxs}
        rows.append({"broad_region": region, "cell_group": cgroup, "n_cells": len(gg), **chunk})
    agg = pd.DataFrame(rows).sort_values(["broad_region", "cell_group"])
    tout = FIG_DIR / "neurofil_ncam_mean_log1p_by_celltype_region.csv"
    agg.to_csv(tout, index=False)
    print("Wrote", tout)

    # Small multi-panel bar: EEC L vs EEC I in ileum for each gene
    ile = agg[agg["broad_region"].eq("ileum")]
    focus = ile[ile["cell_group"].isin(["EEC L", "EEC I Cells (CCK+)", "EEC I", "Glia"])].copy()
    if focus.empty:
        print("No ileum rows for EEC L / EEC I / Glia in expression summary")
        return
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.2 * len(idxs), 3.2), constrained_layout=True)
    if len(idxs) == 1:
        axes = [axes]
    for ax, g in zip(axes, idxs):
        d = focus.pivot_table(index="cell_group", values=g, aggfunc="mean")
        d.plot(kind="barh", ax=ax, legend=False, color=NATURE_CYCLE[0])
        ax.set_title(g, fontsize=7)
        ax.set_xlabel("mean log1p expr")
    fig.suptitle("Ileum: mean log1p (subset of cells)", fontsize=7)
    save_pdf(fig, FIG_DIR / "fig_neuropod_neurofil_ncam_expr_ileum_bars.pdf")


plot_neurofil_ncam_expr_by_eec()


### 2.2 Follicle-associated resident macrophages ↔ T subsets
Pairs include **CD4 Tr1**, **CD4 Tfr**, **CD4 Tfh**, **CD4 pTreg**, **CD4 Th17**, **CD4 tTreg**, **NKT Cells**, **Gamma Delta T Cells** (LIANA naming).

In [6]:
farm_tables = {}
for t in sorted(T_WITH_FARM):
    m = bidirectional_mask(dfw, FARM, {t})
    sub = dfw.loc[m].sort_values("lrscore", ascending=False)
    farm_tables[t] = sub
    print(f"\n=== FARM <-> {t} ===  n={len(sub)}")
    if len(sub):
        display(sub.head(12))
        na = sub[neuro_axon_mask(sub)]
        if len(na):
            print("  neuro/axon hits:")
            display(na.head(15))


=== FARM <-> CD4 Tfh ===  n=8978


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
5111,Follicle Associated Resident Macrophages,CD4 Tfh,HLA-B,CD3D,250.954330,0.0,0.001132,0.979237
284,Follicle Associated Resident Macrophages,CD4 Tfh,HLA-DRA,CD4,737.163900,0.0,0.003098,0.973029
11371,Follicle Associated Resident Macrophages,CD4 Tfh,LGALS1,PTPRC,161.609570,0.0,0.000803,0.972883
2295,Follicle Associated Resident Macrophages,CD4 Tfh,HLA-DPA1,CD4,386.748080,0.0,0.003609,0.962949
2353,Follicle Associated Resident Macrophages,CD4 Tfh,HLA-DRB1,CD4,381.890600,0.0,0.003080,0.962718
24291,Follicle Associated Resident Macrophages,CD4 Tfh,LGALS1,CD69,111.552420,0.0,0.000544,0.961529
796,CD4 Tfh,Follicle Associated Resident Macrophages,COPA,CD74,812.396700,0.0,0.000640,0.959653
3168,Follicle Associated Resident Macrophages,CD4 Tfh,HLA-DPB1,CD4,322.042630,0.0,0.003073,0.959451
29060,Follicle Associated Resident Macrophages,CD4 Tfh,HMGB1,CXCR4,100.721054,0.0,0.000373,0.956796
1312,CD4 Tfh,Follicle Associated Resident Macrophages,APP,CD74,810.964000,0.0,0.000184,0.947623


  neuro/axon hits:


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
11371,Follicle Associated Resident Macrophages,CD4 Tfh,LGALS1,PTPRC,161.609570,0.000,0.000803,0.972883
25025,Follicle Associated Resident Macrophages,CD4 Tfh,LGALS9,PTPRC,109.532196,0.000,0.001918,0.940010
27567,Follicle Associated Resident Macrophages,CD4 Tfh,MRC1,PTPRC,103.735000,0.000,0.002167,0.919246
245435,Follicle Associated Resident Macrophages,CD4 Tfh,CALM1,PTPRA,27.956272,1.000,0.000112,0.837456
204075,CD4 Tfh,Follicle Associated Resident Macrophages,CALM1,PTPRA,31.437893,1.000,0.000054,0.781293
217094,CD4 Tfh,Follicle Associated Resident Macrophages,LGALS1,PTPRC,30.251638,1.000,0.000003,0.699040
856283,CD4 Tfh,Follicle Associated Resident Macrophages,SEMA4D,PLXNB2,11.452521,0.000,0.000703,0.687619
218427,CD4 Tfh,Follicle Associated Resident Macrophages,LGALS9,PTPRC,30.122974,0.512,0.000036,0.682437
883303,CD4 Tfh,Follicle Associated Resident Macrophages,SEMA4D,CD72,11.164046,0.000,0.000977,0.675299
146133,Follicle Associated Resident Macrophages,CD4 Tfh,CD22,PTPRC,97.465120,0.000,0.000074,0.671316



=== FARM <-> CD4 Tfr ===  n=8978


...

,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
22528,Follicle Associated Resident Macrophages,Gamma Delta T Cells,LGALS1,PTPRC,116.021850,0.000,0.000427,0.963168
62543,Follicle Associated Resident Macrophages,Gamma Delta T Cells,LGALS9,PTPRC,63.944470,0.000,0.001019,0.919492
74811,Follicle Associated Resident Macrophages,Gamma Delta T Cells,MRC1,PTPRC,58.147274,0.000,0.001151,0.892439
173384,Gamma Delta T Cells,Follicle Associated Resident Macrophages,LGALS1,PTPRC,34.855920,1.000,0.000021,0.852103
268206,Follicle Associated Resident Macrophages,Gamma Delta T Cells,CALM1,PTPRA,26.337925,1.000,0.000081,0.814118
181397,Gamma Delta T Cells,Follicle Associated Resident Macrophages,CALM1,PTPRA,33.799366,1.000,0.000058,0.787839
208647,Gamma Delta T Cells,Follicle Associated Resident Macrophages,LGALS9,PTPRC,31.043173,0.284,0.000079,0.761318
1233251,Gamma Delta T Cells,Follicle Associated Resident Macrophages,SEMA4D,PLXNB2,8.622633,0.000,0.000476,0.644294
1285002,Gamma Delta T Cells,Follicle Associated Resident Macrophages,SEMA4D,CD72,8.334158,0.000,0.000661,0.631183
245613,Gamma Delta T Cells,Follicle Associated Resident Macrophages,HSPA8,ADRB2,27.941044,0.039,0.000104,0.604972



=== FARM <-> NKT Cells ===  n=8978


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
415,Follicle Associated Resident Macrophages,NKT Cells,B2M,KLRD1,683.21560,0.0,0.001314,0.978536
420,Follicle Associated Resident Macrophages,NKT Cells,B2M,KLRC2,680.44190,0.0,0.002448,0.976302
436,Follicle Associated Resident Macrophages,NKT Cells,B2M,KLRC1,676.82623,0.0,0.001438,0.971948
6519,Follicle Associated Resident Macrophages,NKT Cells,HLA-B,CD3D,220.36725,0.0,0.000506,0.969249
19699,Follicle Associated Resident Macrophages,NKT Cells,LGALS1,PTPRC,122.41185,0.0,0.000480,0.965181
638,Follicle Associated Resident Macrophages,NKT Cells,HLA-DRA,LAG3,732.79630,0.0,0.002634,0.962180
7096,Follicle Associated Resident Macrophages,NKT Cells,HLA-B,KLRD1,210.80762,0.0,0.001382,0.961048
878,NKT Cells,Follicle Associated Resident Macrophages,COPA,CD74,812.05940,0.0,0.000577,0.957581
28893,Follicle Associated Resident Macrophages,NKT Cells,LGALS1,CD69,101.13844,0.0,0.000424,0.956643
2343,Follicle Associated Resident Macrophages,NKT Cells,HLA-DPA1,LAG3,382.38050,0.0,0.003069,0.948260


  neuro/axon hits:


,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,spec_weight,lrscore
19699,Follicle Associated Resident Macrophages,NKT Cells,LGALS1,PTPRC,122.411850,0.000,0.000480,0.965181
52526,Follicle Associated Resident Macrophages,NKT Cells,LGALS9,PTPRC,70.334470,0.000,0.001145,0.923704
108434,NKT Cells,Follicle Associated Resident Macrophages,LGALS1,PTPRC,47.053593,0.000,0.000067,0.911788
61395,Follicle Associated Resident Macrophages,NKT Cells,MRC1,PTPRC,64.537280,0.000,0.001294,0.897908
236631,Follicle Associated Resident Macrophages,NKT Cells,CALM1,PTPRA,28.625032,1.000,0.000124,0.844707
100065,NKT Cells,Follicle Associated Resident Macrophages,CALM1,PTPRA,49.421658,0.000,0.000087,0.819412
507436,Follicle Associated Resident Macrophages,NKT Cells,RTN4,LINGO1,17.034143,0.000,0.001094,0.774813
208785,NKT Cells,Follicle Associated Resident Macrophages,LGALS9,PTPRC,31.025623,0.292,0.000079,0.760366
849053,NKT Cells,Follicle Associated Resident Macrophages,VEGFB,NRP1,11.539065,0.000,0.000654,0.715325
581840,NKT Cells,Follicle Associated Resident Macrophages,GNAI2,UNC5B,15.430880,0.000,0.000283,0.640801


## 3. Figures (Nature-style PDF)
Bar panels: top interactions by `lrscore` per question group. **All figure text in black**; bars use the Nature/Wong palette; **Helvetica** (fallback Arial / DejaVu). **Direction**: LIANA uses **source** and **target**; **ligand** is on **source** cells, **receptor** on **target** cells. Row labels list **ligand — receptor** (not an arrow). Where tables pool **bidirectional** masks, each plotted value is the **max** `lrscore` over **both** orientations (e.g. Glia→epithelium and epithelium→Glia). Notes are printed on the PDFs (footer or second line of title).

In [7]:
# --- Fig 1: Epithelium / progenitors ↔ Glia (10 panels, a–j) ---
glia_items = list(glia_tables.items())
nrows, ncols = 5, 2
fig1, axes1 = plt.subplots(nrows, ncols, figsize=(7.2, 11.0), constrained_layout=True)
axes1 = axes1.ravel()
letters = "abcdefghijklmnopqrst"
for i, (name, sub) in enumerate(glia_items):
    ax = axes1[i]
    panel_label(ax, letters[i] + ")")
    hbar_top_lr(sub, ax, title=name.replace(" <-> ", " ↔ "), n=10)
for j in range(len(glia_items), len(axes1)):
    axes1[j].set_axis_off()
fig1.suptitle(
    "Ileum LIANA: epithelial / progenitor <-> Glia (top lrscore)\n" + LIANA_NOTE_GLIA_BARS,
    fontsize=7,
    color=NATURE_WONG["black"],
    y=1.03,
)
save_pdf(fig1, FIG_DIR / "fig1_epithelium_glia_lrscore.pdf")

# --- Fig 2: FARM ↔ lymphocyte subsets (8 panels) ---
farm_items = sorted(farm_tables.items(), key=lambda x: x[0])
fig2, axes2 = plt.subplots(4, 2, figsize=(7.2, 9.2), constrained_layout=True)
axes2 = axes2.ravel()
for i, (tname, sub) in enumerate(farm_items):
    ax = axes2[i]
    panel_label(ax, letters[i] + ")")
    short = tname.replace("CD4 ", "").replace("Gamma Delta T Cells", "γδ T").replace("Cells", "").strip()
    hbar_top_lr(sub, ax, title=f"FARM ↔ {short}", n=10)
for j in range(len(farm_items), len(axes2)):
    axes2[j].set_axis_off()
fig2.suptitle(
    "Ileum LIANA: Follicle-associated resident macrophages <-> T / NK (top lrscore)\n" + LIANA_NOTE_FARM_BARS,
    fontsize=7,
    color=NATURE_WONG["black"],
    y=1.03,
)
save_pdf(fig2, FIG_DIR / "fig2_farm_lymphocyte_lrscore.pdf")

print("Figure directory:", FIG_DIR.resolve())

/var/folders/zc/5fly3ly908xb0pd8zy6m6bj80000gn/T/ipykernel_96935/3147547659.py:59: UserWarning: Glyph 8596 (\N{LEFT RIGHT ARROW}) missing from font(s) Helvetica.
  fig.savefig(
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
/var/folders/zc/5fly3ly908xb0pd8zy6m6bj80000gn/T/ipykernel_96935/3147547659.py:59: UserWarning: Glyph 8596 (\N{LEFT RIGHT ARROW}) missing from font(s) Helvetica.
  fig.savefig(


Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig1_epithelium_glia_lrscore.pdf


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp


Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig2_farm_lymphocyte_lrscore.pdf
Figure directory: /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures


In [8]:
# --- Fig 3: Neurotransmitter / axon keyword hits (epithelium–Glia, pooled) ---
na_parts = []
for gname, sub in glia_tables.items():
    hit = sub[neuro_axon_mask(sub)]
    if len(hit):
        h = hit.copy()
        h["group"] = gname
        na_parts.append(h)
if na_parts:
    na_all = pd.concat(na_parts, ignore_index=True)
    na_all = na_all.sort_values("lrscore", ascending=False)
    na_dedup = na_all.drop_duplicates(subset=["ligand_complex", "receptor_complex"]).head(22)
else:
    na_dedup = pd.DataFrame(columns=USECOLS + ["group"])

fig3, ax3 = plt.subplots(figsize=(3.5, 6.5))
panel_label(ax3, "a)")
hbar_top_lr(na_dedup, ax3, title="Neuro / axon keyword hits (any epithelium–Glia group)", n=min(22, len(na_dedup)))
fig3.suptitle(
    "Ileum LIANA: neurotransmitter & axon-guidance keywords\n" + LIANA_NOTE_GLIA_BARS,
    fontsize=7,
    color=NATURE_WONG["black"],
    y=1.03,
)
save_pdf(fig3, FIG_DIR / "fig3_neuro_axon_epithelium_glia.pdf")

1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp


Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig3_neuro_axon_epithelium_glia.pdf


### Differential CCC (epithelium ↔ Glia)
Heatmaps: **columns** = epithelial subset context (Fig. 1 groups, including **BEST4 colonocytes** separately from **BEST4 enterocytes**, and **Lower Villus enterocytes** as a **control** column). **Row ranking** uses variability **excluding** the **Villus bottom** column so control-driven pairs are not promoted. **PYY / SST / NTS / TPH1** outgoing interactions are collapsed to **one canonical L–R per hormone** before plotting. Two PDFs: **all** pairs vs **neuro / axon keyword** hits only. **Scatter / delta** (below) use **neuro / axon** colouring between two epithelial contexts.

In [9]:
# Short x-axis labels (epithelial subset context); Glia appears in figure titles only.
EPI_CONTEXT_LABELS = {
    "EEC L <-> Glia": "EEC L",
    "EEC I (CCK+) <-> Glia": "EEC I (CCK+)",
    "EEC (mature) <-> Glia": "EEC (mature)",
    "BEST4 Enterocyte <-> Glia": "BEST4 enterocyte",
    "BEST4 Colonocytes <-> Glia": "BEST4 colonocyte",
    "Tuft <-> Glia": "Tuft",
    "EEC Progenitors <-> Glia": "EEC prog.",
    "Tuft Progenitors <-> Glia": "Tuft prog.",
    "Secretory Progenitors <-> Glia": "Secr. prog.",
    "Villus-bottom enterocytes <-> Glia": "Villus bottom",
}
# Heatmap row ranking excludes this column (control epithelium; not used to pick top variable pairs)
GLIA_HEATMAP_CTRL_COL = "Villus bottom"


def wide_epithelium_glia(glia_tables: dict, neuro_only: bool) -> pd.DataFrame:
    parts = []
    for gname, sub in glia_tables.items():
        s = sub
        if neuro_only:
            s = s[neuro_axon_mask(s)]
        if s.empty:
            continue
        m = max_lr_per_pair(s)
        col = EPI_CONTEXT_LABELS.get(gname, shorten_label(gname, 18))
        parts.append(m.set_index(["ligand_complex", "receptor_complex"])["lrscore"].rename(col))
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, axis=1)


def plot_glia_epithelium_diff_heatmap(
    wide: pd.DataFrame,
    *,
    axis_title: str,
    suptitle: str,
    outfile: Path,
    top_n: int = 40,
    exclude_cols_for_ranking: set[str] | None = None,
    rank_by_column: str | None = None,
    direction_note: str | None = None,
) -> None:
    if wide.empty or wide.shape[1] < 1:
        print("Skipping heatmap (no data):", outfile.name)
        return
    if rank_by_column is not None and rank_by_column in wide.columns:
        ranked = wide[rank_by_column].dropna().sort_values(ascending=False)
        if ranked.empty:
            print("Skipping heatmap (no non-null values in rank column):", rank_by_column, outfile.name)
            return
        n = min(top_n, len(ranked))
        heat = wide.loc[ranked.head(n).index]
    else:
        exc = exclude_cols_for_ranking or set()
        cols_rank = [c for c in wide.columns if c not in exc]
        if not cols_rank:
            cols_rank = list(wide.columns)
        wide_rank = wide[cols_rank]
        rank_var = wide_rank.std(axis=1, skipna=True).dropna().sort_values(ascending=False)
        if rank_var.empty:
            rank_var = wide_rank.max(axis=1, skipna=True).sort_values(ascending=False)
        n = min(top_n, len(rank_var))
        heat = wide.loc[rank_var.head(n).index]
    note = direction_note if direction_note is not None else LIANA_NOTE_GLIA_HEATMAP
    fig_h, ax_h = plt.subplots(figsize=(7.2, 8.8))
    fig_h.subplots_adjust(bottom=0.2, top=0.86)
    panel_label(ax_h, "a)")
    data = np.ma.masked_invalid(heat.values.astype(float))
    vmin_h = float(np.nanmin(heat.values))
    vmax_h = float(np.nanmax(heat.values))
    if vmin_h == vmax_h:
        vmax_h = vmin_h + 1e-9
    im = ax_h.imshow(data, aspect="auto", cmap=NATURE_CMAP, vmin=vmin_h, vmax=vmax_h)
    ax_h.set_yticks(np.arange(len(heat)))
    ax_h.set_yticklabels(
        [shorten_label(f"{a} — {b}") for a, b in heat.index],
        fontsize=5,
        color=NATURE_WONG["black"],
    )
    ax_h.set_xticks(np.arange(heat.shape[1]))
    ax_h.set_xticklabels(list(heat.columns), rotation=45, ha="right", fontsize=6, color=NATURE_WONG["black"])
    ax_h.set_xlabel("Epithelial subset (vs Glia)", color=NATURE_WONG["black"], fontsize=7)
    ax_h.set_title(axis_title, fontsize=7, color=NATURE_WONG["black"])
    cbar = fig_h.colorbar(im, ax=ax_h, fraction=0.035, pad=0.02)
    cbar.ax.tick_params(colors=NATURE_WONG["black"])
    cbar.set_label("lrscore (unitless)", color=NATURE_WONG["black"], fontsize=6)
    for t in cbar.ax.get_yticklabels():
        t.set_color(NATURE_WONG["black"])
    fig_h.suptitle(suptitle, fontsize=7, color=NATURE_WONG["black"], y=0.98)
    fig_h.text(
        0.5,
        0.06,
        note,
        ha="center",
        va="top",
        fontsize=5.5,
        color=NATURE_WONG["black"],
        transform=fig_h.transFigure,
    )
    save_pdf(fig_h, outfile)


wide_all = wide_epithelium_glia(glia_tables, neuro_only=False)
plot_glia_epithelium_diff_heatmap(
    wide_all,
    axis_title="Glia: variable L–R pairs across epithelial contexts (max lrscore, all pairs)",
    suptitle="Differential CCC (ileum LIANA): Glia vs epithelial subsets",
    outfile=FIG_DIR / "fig_epithelium_glia_diff_heatmap_all.pdf",
    exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
)

wide_na = wide_epithelium_glia(glia_tables, neuro_only=True)
plot_glia_epithelium_diff_heatmap(
    wide_na,
    axis_title="Glia: variable L–R pairs (neuro / axon keywords only)",
    suptitle="Differential CCC (ileum LIANA): Glia vs epithelial subsets (neuro / axon)",
    outfile=FIG_DIR / "fig_epithelium_glia_diff_heatmap_neuro_axon.pdf",
    exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
)

# --- Glia / epithelium: scatter & delta (neuro / axon stratification) ---
SC_A, SC_B = "EEC (mature)", "Tuft"
sc_cols = [c for c in (SC_A, SC_B) if c in wide_all.columns]
if len(sc_cols) < 2:
    sc_cols = list(wide_all.columns)[:2]
if len(sc_cols) >= 2:
    ca, cb = sc_cols[0], sc_cols[1]
    scat = wide_all[[ca, cb]].dropna(how="any").reset_index()
    scat["neuro_axon"] = neuro_axon_mask(scat)
    fig_es, ax_es = plt.subplots(figsize=(3.5, 3.5))
    panel_label(ax_es, "a)")
    oth = ~scat["neuro_axon"]
    ax_es.scatter(
        scat.loc[oth, ca],
        scat.loc[oth, cb],
        s=10,
        c=NATURE_WONG["sky_blue"],
        edgecolors=NATURE_WONG["black"],
        linewidths=0.35,
        alpha=0.55,
        label="Other L–R",
    )
    ax_es.scatter(
        scat.loc[scat["neuro_axon"], ca],
        scat.loc[scat["neuro_axon"], cb],
        s=14,
        c=NATURE_WONG["vermillion"],
        edgecolors=NATURE_WONG["black"],
        linewidths=0.35,
        alpha=0.9,
        label="Neuro / axon keyword",
    )
    mx = float(max(scat[ca].max(), scat[cb].max()))
    ax_es.plot([0, mx], [0, mx], color=NATURE_WONG["black"], linewidth=0.6, linestyle="--", alpha=0.65)
    ax_es.set_xlabel(f"lrscore Glia <-> {ca} (unitless)", color=NATURE_WONG["black"])
    ax_es.set_ylabel(f"lrscore Glia <-> {cb} (unitless)", color=NATURE_WONG["black"])
    ax_es.set_title(f"Matched pairs: {ca} vs {cb}", fontsize=7, color=NATURE_WONG["black"])
    leg_es = ax_es.legend(frameon=False, loc="upper left", fontsize=6, handlelength=1.2)
    for tx in leg_es.get_texts():
        tx.set_color(NATURE_WONG["black"])
    ax_es.tick_params(colors=NATURE_WONG["black"])
    fig_es.subplots_adjust(bottom=0.2)
    fig_es.suptitle(
        "Differential CCC (Glia): epithelial contexts (neuro / axon)",
        fontsize=7,
        color=NATURE_WONG["black"],
        y=1.06,
    )
    fig_es.text(
        0.5,
        0.03,
        LIANA_NOTE_SHORT,
        ha="center",
        va="bottom",
        fontsize=5,
        color=NATURE_WONG["black"],
        transform=fig_es.transFigure,
    )
    save_pdf(fig_es, FIG_DIR / "fig_epithelium_glia_diff_scatter_neuro_axon.pdf")

    scat["delta"] = scat[ca] - scat[cb]
    top_ed = scat.reindex(scat["delta"].abs().sort_values(ascending=False).index).head(18)
    fig_ed, ax_ed = plt.subplots(figsize=(3.5, 5.5))
    panel_label(ax_ed, "b)")
    ye = np.arange(len(top_ed))
    bar_c = [
        NATURE_WONG["vermillion"] if neuro_axon_mask(top_ed.iloc[[i]]).iloc[0] else NATURE_WONG["sky_blue"]
        for i in range(len(top_ed))
    ]
    ax_ed.barh(ye, top_ed["delta"], color=bar_c, edgecolor=NATURE_WONG["black"], linewidth=0.35)
    labs_ed = [
        shorten_label(f"{a} — {b}") for a, b in zip(top_ed["ligand_complex"], top_ed["receptor_complex"])
    ]
    ax_ed.set_yticks(ye)
    ax_ed.set_yticklabels(labs_ed, fontsize=5.5, color=NATURE_WONG["black"])
    ax_ed.invert_yaxis()
    ax_ed.set_xlabel(f"Δ lrscore ({ca} − {cb}) (unitless)", color=NATURE_WONG["black"])
    ax_ed.set_title("Largest |Δ| (Glia-linked)", fontsize=7, color=NATURE_WONG["black"])
    ax_ed.axvline(0, color=NATURE_WONG["black"], linewidth=0.5)
    ax_ed.tick_params(colors=NATURE_WONG["black"])
    ax_ed.spines["top"].set_visible(False)
    ax_ed.spines["right"].set_visible(False)
    leg_ed = ax_ed.legend(
        handles=[
            Patch(facecolor=NATURE_WONG["vermillion"], edgecolor=NATURE_WONG["black"], label="Neuro / axon keyword"),
            Patch(facecolor=NATURE_WONG["sky_blue"], edgecolor=NATURE_WONG["black"], label="Other"),
        ],
        frameon=False,
        loc="lower right",
        fontsize=6,
    )
    for tx in leg_ed.get_texts():
        tx.set_color(NATURE_WONG["black"])
    fig_ed.subplots_adjust(bottom=0.18)
    fig_ed.suptitle(
        "Differential CCC (Glia): epithelial contrast (neuro / axon)",
        fontsize=7,
        color=NATURE_WONG["black"],
        y=1.02,
    )
    fig_ed.text(
        0.5,
        0.02,
        LIANA_NOTE_SHORT,
        ha="center",
        va="bottom",
        fontsize=5,
        color=NATURE_WONG["black"],
        transform=fig_ed.transFigure,
    )
    save_pdf(fig_ed, FIG_DIR / "fig_epithelium_glia_diff_delta_neuro_axon.pdf")
else:
    print("Skipping epithelial scatter/delta: need ≥2 columns in wide_all")

1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp


Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_diff_heatmap_all.pdf


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp


Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_diff_heatmap_neuro_axon.pdf
Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_diff_scatter_neuro_axon.pdf
Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_diff_delta_neuro_axon.pdf


### Epithelium–Glia: BEST4 CFTR split & villus columns (de novo LIANA)
The ileum LIANA CSV does not separate **BEST4 Enterocytes** by **CFTR** expression. This block loads a **memory subset** of the integrated atlas (ileum cells), assigns **CFTR-high** vs **not-high** among BEST4 enterocytes (median **log1p CFTR**), runs **`liana`** on small **Glia + epithelium** subsets, and merges results with **`wide_all`** from the precomputed table.
**`rank_aggregate` output** lists **all** sender–receiver pairs among the `groupby` labels, including **within-group** rows where **`source == target`** (e.g. BEST4_Ent_CFTRhi → BEST4_Ent_CFTRhi). Those are **not** used for the Glia heatmap columns: **`_lr_series_from_liana`** keeps only **Glia ↔ epithelial** pairs (**no** autocrine / within-epithelium-only).
- **Heatmaps (all pairs + neuro/axon, CFTR split)**: same matrices as `fig_epithelium_glia_diff_heatmap_all.pdf` and `fig_epithelium_glia_diff_heatmap_neuro_axon.pdf`, but **BEST4 enterocyte** is replaced by **BEST4 CFTR-hi** and **BEST4 CFTR-lo** → `fig_epithelium_glia_diff_heatmap_all_cftr_split.pdf`, `fig_epithelium_glia_diff_heatmap_neuro_axon_cftr_split.pdf`.
- **Heatmap (CFTR focus)**: same epithelial contexts as §differential heatmaps, but **BEST4 enterocyte** is replaced by **BEST4 CFTR-hi** and **BEST4 CFTR-lo**; **rows ranked by lrscore in the CFTR-hi column** (Glia ↔ CFTR-high BEST4).
- **Heatmap (curated gene panel)**: same **`wide_cftr`** matrix, but **only L–R pairs** whose `ligand_complex` / `receptor_complex` strings hit **human (HGNC)** symbols including: epithelial/ion (**BEST4, CFTR, OTOP2, CA7, GUCA2A/B, GUCY2C**); presynaptic (**SYN1, PCLO, BSN, UNC13B, RIMS2, SYT3, NOTCH2**); postsynaptic/scaffold (**HOMER3, DLG4**); **NRXN/NLGN**; **EFNB/EPH**; **LRRTM**, **LRRC4**; IgCAM-type (**NCAM1, L1CAM, CADM1–3, PTPRF, DSCAM, OPCML**); glutamate vesicular transporters (**SLC17A7, SLC17A6**); neurotrophin/RET (**NTRK1, GFRA1–4, RET, NGF, ARTN, GDNF, NRTN**); axon guidance (**SEMA, PLXN, NRP, NTN, DCC, UNC5, SLIT, ROBO** families as listed in the notebook). Ranked by **CFTR-hi** when available → `fig_epithelium_glia_neuropod_epithelial_gene_panel.pdf`.
- **Heatmap (villus column)**: **Glia** vs **BEST4 colonocytes**, **Lower villus enterocytes**, **Villus tip enterocytes** only (de novo LIANA on those four types).
Requires `pip install liana` and sufficient cells per group (≥ ~5 per identity).

In [10]:
try:
    import liana as li
except ImportError as exc:
    raise ImportError("Install liana: pip install liana") from exc

COL_BEST4_ENT = EPI_CONTEXT_LABELS["BEST4 Enterocyte <-> Glia"]  # "BEST4 enterocyte"
FOCUS_CFTR_COL = "BEST4 CFTR-hi"


def _glia_epi_block(df: pd.DataFrame, epithelial: str) -> pd.DataFrame:
    """Rows where Glia <-> epithelial (bidirectional). Excludes within-group / autocrine (source==target==epi)."""
    if df is None or df.empty:
        return pd.DataFrame()
    s = df["source"].astype(str)
    t = df["target"].astype(str)
    epi = str(epithelial)
    m = ((s == epi) & (t == "Glia")) | ((s == "Glia") & (t == epi))
    return df.loc[m].copy()


def _fill_ligand_receptor_complex_from_simple(sub: pd.DataFrame) -> pd.DataFrame:
    """LIANA often has ligand/receptor; ligand_complex may be empty — fill for indexing."""
    if sub.empty:
        return sub
    out = sub.copy()
    if "ligand_complex" in out.columns and "ligand" in out.columns:
        lc = out["ligand_complex"]
        m = lc.isna() | (lc.astype(str).str.strip().eq("")) | (lc.astype(str).str.lower().eq("nan"))
        out.loc[m, "ligand_complex"] = out.loc[m, "ligand"].astype(str)
    if "receptor_complex" in out.columns and "receptor" in out.columns:
        rc = out["receptor_complex"]
        m = rc.isna() | (rc.astype(str).str.strip().eq("")) | (rc.astype(str).str.lower().eq("nan"))
        out.loc[m, "receptor_complex"] = out.loc[m, "receptor"].astype(str)
    return out


def _lr_series_from_liana(df: pd.DataFrame, epithelial: str, col_name: str) -> pd.Series:
    """Max lrscore per L-R for Glia <-> epithelial only (ignores rank_aggregate within-group rows)."""
    sub = _glia_epi_block(df, epithelial)
    if sub.empty:
        return pd.Series(dtype=float, name=col_name)
    if "lrscore" not in sub.columns:
        for alt in ("aggregate_rank", "lr_mean", "mean_rank"):
            if alt in sub.columns:
                sub = sub.rename(columns={alt: "lrscore"})
                break
    if "lrscore" not in sub.columns:
        return pd.Series(dtype=float, name=col_name)
    sub = _fill_ligand_receptor_complex_from_simple(sub)
    sub = reduce_canonical_neuropeptide_outgoing(sub)
    m = max_lr_per_pair(sub)
    return m.set_index(["ligand_complex", "receptor_complex"])["lrscore"].rename(col_name)


TISSUE_COL = "tissue_ontology_term"
ad_back = sc.read_h5ad(INTEGRATED, backed="r")
obs = ad_back.obs
mask_ile = obs[TISSUE_COL].astype(str).str.lower().str.contains("ileum", na=False)
types_cftr = {"Glia", "BEST4 Enterocytes"}
mask_ct = obs["hgca_celltype_v1"].astype(str).isin(types_cftr)
ix = np.where((mask_ile & mask_ct).values)[0]
print("Ileum Glia+BEST4 cells for CFTR split:", len(ix))
ad_cf = ad_back[ix].to_memory()
if hasattr(ad_back, "file") and ad_back.file is not None:
    ad_back.file.close()

if "gene_symbol" in ad_cf.var.columns:
    ad_cf.var_names = ad_cf.var["gene_symbol"].astype(str).values
    ad_cf.var_names_make_unique()
if "CFTR" not in ad_cf.var_names:
    raise KeyError("CFTR not in var_names — check gene_symbol mapping")
print(
    "ad_cf var_names use gene symbols (post-mapping): n_var =",
    ad_cf.n_vars,
    "; CFTR present:",
    bool(np.any(ad_cf.var_names.astype(str).str.upper().values == "CFTR")),
)

xcf = ad_cf[:, "CFTR"].X
xcf = xcf.toarray().ravel() if hasattr(xcf, "toarray") else np.asarray(xcf).ravel()
xcf = np.log1p(np.maximum(xcf.astype(float), 0.0))
b4 = ad_cf.obs["hgca_celltype_v1"].astype(str).values == "BEST4 Enterocytes"
thr = float(np.median(xcf[b4])) if b4.sum() else 0.0
cf_hi = np.zeros(ad_cf.n_obs, dtype=bool)
cf_hi[b4] = xcf[b4] >= thr
ad_cf.obs["ccc_group"] = ad_cf.obs["hgca_celltype_v1"].astype(str)
ad_cf.obs.loc[b4 & cf_hi, "ccc_group"] = "BEST4_Ent_CFTRhi"
ad_cf.obs.loc[b4 & ~cf_hi, "ccc_group"] = "BEST4_Ent_CFTRlo"
print("BEST4 ileum: CFTR-hi", int(cf_hi[b4].sum()), "CFTR-lo", int((b4 & ~cf_hi).sum()), "median log1p thr", round(thr, 4))

df_hi = pd.DataFrame()
df_lo = pd.DataFrame()
for lab, tag in [("BEST4_Ent_CFTRhi", "hi"), ("BEST4_Ent_CFTRlo", "lo")]:
    mkeep = ad_cf.obs["ccc_group"].astype(str).isin(["Glia", lab])
    sub = ad_cf[mkeep].copy()
    vc = sub.obs["ccc_group"].value_counts()
    print(f"LIANA subset {tag}:", vc.to_dict())
    if vc.min() < 5:
        print(f"  skip {tag}: <5 cells in a group")
        continue
    try:
        out = li.mt.rank_aggregate(
            sub,
            groupby="ccc_group",
            resource_name="consensus",
            expr_prop=0.1,
            use_raw=False,
            inplace=False,
            verbose=False,
        )
        if isinstance(out, tuple):
            out = out[0]
        dfo = out if isinstance(out, pd.DataFrame) else pd.DataFrame()
        if tag == "hi":
            df_hi = dfo
        else:
            df_lo = dfo
    except Exception as e:
        print(f"LIANA CFTR {tag} failed:", e)

print(
    "Note: rank_aggregate includes within-group (source==target) rows in df_hi/df_lo; "
    "heatmap Glia columns use only Glia↔epithelial pairs (_glia_epi_block), not autocrine."
)

for _lab, _df in (("df_hi", df_hi), ("df_lo", df_lo)):
    if _df is not None and len(_df):
        print(f"--- {_lab}: shape={_df.shape}")
        print("  columns:", list(_df.columns[:min(25, len(_df.columns))]), "..." if len(_df.columns) > 25 else "")
        _need = {"ligand_complex", "receptor_complex", "source", "target"}
        print("  has", _need, "?", {c: c in _df.columns for c in _need})
        if "ligand" in _df.columns and "ligand_complex" in _df.columns:
            _n = _df["ligand_complex"].isna() | (_df["ligand_complex"].astype(str).str.strip().eq(""))
            print("  ligand_complex empty/NaN rows:", int(_n.sum()), "of", len(_df))
        _blk = _glia_epi_block(_df, "BEST4_Ent_CFTRhi" if _lab == "df_hi" else "BEST4_Ent_CFTRlo")
        if len(_blk):
            print("  sample Glia↔epi row (ligand_complex, receptor_complex):", _blk.iloc[0][["ligand_complex", "receptor_complex"]].to_dict())

s_hi = _lr_series_from_liana(df_hi, "BEST4_Ent_CFTRhi", FOCUS_CFTR_COL)
s_lo = _lr_series_from_liana(df_lo, "BEST4_Ent_CFTRlo", "BEST4 CFTR-lo")

wide_cftr = wide_all.drop(columns=[COL_BEST4_ENT], errors="ignore").copy()
wide_cftr = wide_cftr.join(s_hi, how="outer")
wide_cftr = wide_cftr.join(s_lo, how="outer")

# Same as differential heatmaps (all pairs / neuro-axon) but with BEST4 CFTR-hi/lo columns
wide_all_cftr = wide_all.drop(columns=[COL_BEST4_ENT], errors="ignore").copy()
wide_all_cftr = wide_all_cftr.join(s_hi, how="outer").join(s_lo, how="outer")
plot_glia_epithelium_diff_heatmap(
    wide_all_cftr,
    axis_title="Glia: variable L–R pairs (BEST4 CFTR-hi/lo + other subsets; max lrscore, all pairs)",
    suptitle="Differential CCC (ileum LIANA): Glia vs epithelial subsets (BEST4 CFTR-high / CFTR-low)",
    outfile=FIG_DIR / "fig_epithelium_glia_diff_heatmap_all_cftr_split.pdf",
    exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
)
wide_na_cftr = wide_na.drop(columns=[COL_BEST4_ENT], errors="ignore").copy()
wide_na_cftr = wide_na_cftr.join(s_hi, how="outer").join(s_lo, how="outer")
plot_glia_epithelium_diff_heatmap(
    wide_na_cftr,
    axis_title="Glia: variable L–R pairs (neuro / axon keywords; BEST4 CFTR-hi/lo)",
    suptitle="Differential CCC (ileum LIANA): Glia vs epithelial subsets, neuro/axon (BEST4 CFTR split)",
    outfile=FIG_DIR / "fig_epithelium_glia_diff_heatmap_neuro_axon_cftr_split.pdf",
    exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
)

print("--- wide_cftr (gene-panel input): shape =", wide_cftr.shape)
print("  index type:", type(wide_cftr.index).__name__, "; names:", getattr(wide_cftr.index, "names", None))
if len(wide_cftr):
    _w0 = wide_cftr.index[0]
    print("  first index value repr:", repr(_w0))
    _tmp = wide_cftr.reset_index()
    print("  reset_index() first columns:", list(_tmp.columns[:4]))
    print("  first row L/R from reset_index (cols 0,1):", _tmp.iloc[0, 0], "|", _tmp.iloc[0, 1])
    _probe = (str(_tmp.iloc[0, 0]) + " " + str(_tmp.iloc[0, 1])).upper()
    for _g in ("CFTR", "SYN1", "GUCA2A", "PCLO"):
        print(f"  substring '{_g}' in first-row L+R string:", _g in _probe)
    _hit = sum(
        1
        for i in range(min(5000, len(_tmp)))
        if any(x in (str(_tmp.iloc[i, 0]) + str(_tmp.iloc[i, 1])).upper() for x in ("CFTR", "SYN1", "GUCA2A"))
    )
    print("  rows (first 5000) with CFTR or SYN1 or GUCA2A substring in L+R:", _hit)

if wide_cftr[FOCUS_CFTR_COL].notna().any():
    plot_glia_epithelium_diff_heatmap(
        wide_cftr,
        axis_title="Glia: epithelial contexts (BEST4 split by CFTR); rows = top lrscore in CFTR-hi BEST4",
        suptitle="Differential CCC (ileum): Glia vs epithelium (BEST4 CFTR-high focus)",
        outfile=FIG_DIR / "fig_epithelium_glia_diff_heatmap_best4_cftr_split.pdf",
        exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
        rank_by_column=FOCUS_CFTR_COL,
    )
else:
    print("Skip BEST4 CFTR-split heatmap: no non-null scores in", FOCUS_CFTR_COL)

# --- Epithelium–Glia heatmap: curated BEST4 / ion / neuropod & synaptic gene panel (human HGNC symbols) ---
NEUROPOD_EPITHELIAL_GENE_TOKENS = {
    "BEST4",
    "CFTR",
    "OTOP2",
    "CA7",
    "GUCA2A",
    "GUCA2B",
    "GUCY2C",
    "SYT3",
    "SYN1",
    "PCLO",
    "BSN",
    "UNC13B",
    "RIMS2",
    "NOTCH2",
    "HOMER3",
    "DLG4",
    "SLC17A7",
    "SLC17A6",
    "NRXN1",
    "NRXN2",
    "NRXN3",
    "NLGN1",
    "NLGN2",
    "NLGN3",
    "NLGN4",
    "EFNB1",
    "EFNB2",
    "EFNB3",
    "EPHA2",
    "EPHA4",
    "EPHA7",
    "EPHB2",
    "EPHB4",
    "EPHB6",
    "LRRTM1",
    "LRRTM2",
    "LRRTM3",
    "LRRTM4",
    "LRRC4",
    "LRRC4B",
    "LRRC4C",
    "NCAM1",
    "L1CAM",
    "CADM1",
    "CADM2",
    "CADM3",
    "PTPRF",
    "DSCAM",
    "OPCML",
    "NTRK1",
    "GFRA1",
    "GFRA2",
    "GFRA3",
    "GFRA4",
    "RET",
    "NGF",
    "ARTN",
    "GDNF",
    "NRTN",
    "SEMA3A",
    "SEMA3C",
    "SEMA3F",
    "SEMA4A",
    "PLXNA1",
    "PLXNA2",
    "PLXNA3",
    "PLXNA4",
    "PLXNB1",
    "PLXNB2",
    "PLXNC1",
    "NRP1",
    "NRP2",
    "NTN1",
    "NTN4",
    "DCC",
    "UNC5A",
    "UNC5B",
    "UNC5C",
    "UNC5D",
    "SLIT1",
    "SLIT2",
    "SLIT3",
    "ROBO1",
    "ROBO2",
    "ROBO3",
    "ROBO4",
}
NEUROPOD_EPITHELIAL_GENE_TOKENS = {g.upper() for g in NEUROPOD_EPITHELIAL_GENE_TOKENS}


def _lr_tokens_gene_panel(lig: str, rec: str) -> set[str]:
    s = re.sub(r"[\^]+", " ", f"{lig} {rec}")
    return set(re.findall(r"[A-Za-z0-9]+", s.upper()))


def _gene_hits_lr_string(lig: str, rec: str, tokens: set[str]) -> bool:
    """True if any panel gene appears as a token or as a bounded alphanumeric span in L+R."""
    tks = _lr_tokens_gene_panel(lig, rec)
    if tks & tokens:
        return True
    blob = f" {lig} {rec} "
    blob_u = blob.upper()
    for g in tokens:
        if len(g) < 2:
            continue
        if re.search(r"(?<![A-Z0-9])" + re.escape(g) + r"(?![A-Z0-9])", blob_u):
            return True
    return False


def wide_rows_hitting_gene_panel(wide_df: pd.DataFrame, tokens: set[str]) -> pd.DataFrame:
    """Keep rows whose ligand/receptor strings hit the panel (uses reset_index — robust to index quirks)."""
    if wide_df.empty:
        return wide_df.head(0)
    t = wide_df.reset_index()
    if t.shape[1] < 2:
        return wide_df.head(0)
    lg, rc = t.iloc[:, 0], t.iloc[:, 1]
    mask = []
    for L, R in zip(lg, rc):
        if pd.isna(L) and pd.isna(R):
            mask.append(False)
            continue
        mask.append(_gene_hits_lr_string(str(L), str(R), tokens))
    m = np.array(mask, dtype=bool)
    if not m.any():
        return wide_df.head(0)
    return wide_df.iloc[m]


def _count_liana_glia_epi_panel_hits(df: pd.DataFrame, epithelial: str, tokens: set[str]) -> int:
    sub = _fill_ligand_receptor_complex_from_simple(_glia_epi_block(df, epithelial))
    if sub.empty or not {"ligand_complex", "receptor_complex"}.issubset(sub.columns):
        return 0
    n = 0
    for _, row in sub.iterrows():
        if _gene_hits_lr_string(str(row["ligand_complex"]), str(row["receptor_complex"]), tokens):
            n += 1
    return n


print(
    "  De novo LIANA panel hits (Glia↔epi rows): df_hi =",
    _count_liana_glia_epi_panel_hits(df_hi, "BEST4_Ent_CFTRhi", NEUROPOD_EPITHELIAL_GENE_TOKENS),
    "df_lo =",
    _count_liana_glia_epi_panel_hits(df_lo, "BEST4_Ent_CFTRlo", NEUROPOD_EPITHELIAL_GENE_TOKENS),
)

wide_np = wide_rows_hitting_gene_panel(wide_cftr, NEUROPOD_EPITHELIAL_GENE_TOKENS)
print(
    "Neuropod/BEST4 ion/synaptic panel: L–R pairs matching curated genes:",
    wide_np.shape[0],
)
if wide_np.empty:
    print("Skip gene-panel heatmap: no L–R pairs matched the curated gene list.")
else:
    rk = FOCUS_CFTR_COL if wide_np[FOCUS_CFTR_COL].notna().any() else None
    plot_glia_epithelium_diff_heatmap(
        wide_np,
        axis_title="Glia: epithelial contexts (BEST4 CFTR-hi/lo + subsets); rows = panel genes; rank by CFTR-hi if available",
        suptitle="Epithelium–Glia CCC: BEST4 / guanylate / neuropod & synaptic gene panel",
        outfile=FIG_DIR / "fig_epithelium_glia_neuropod_epithelial_gene_panel.pdf",
        exclude_cols_for_ranking={GLIA_HEATMAP_CTRL_COL},
        rank_by_column=rk,
        top_n=min(40, max(1, wide_np.shape[0])),
    )

# --- Villus / column epithelium: Glia + BEST4 colon + lower villus + villus tip ---
ad_back2 = sc.read_h5ad(INTEGRATED, backed="r")
obs2 = ad_back2.obs
mask_ile2 = obs2[TISSUE_COL].astype(str).str.lower().str.contains("ileum", na=False)
vtypes = {"Glia", "BEST4 Colonocytes", "Lower Villus Enterocytes", "Villus Tip Enterocytes"}
mask_v = obs2["hgca_celltype_v1"].astype(str).isin(vtypes)
ix2 = np.where((mask_ile2 & mask_v).values)[0]
print("Ileum cells for villus-column LIANA:", len(ix2))
ad_v = ad_back2[ix2].to_memory()
if hasattr(ad_back2, "file") and ad_back2.file is not None:
    ad_back2.file.close()
if "gene_symbol" in ad_v.var.columns:
    ad_v.var_names = ad_v.var["gene_symbol"].astype(str).values
    ad_v.var_names_make_unique()

df_v = pd.DataFrame()
if ad_v.n_obs and ad_v.obs["hgca_celltype_v1"].value_counts().min() >= 5:
    try:
        outv = li.mt.rank_aggregate(
            ad_v,
            groupby="hgca_celltype_v1",
            resource_name="consensus",
            expr_prop=0.1,
            use_raw=False,
            inplace=False,
            verbose=False,
        )
        if isinstance(outv, tuple):
            outv = outv[0]
        df_v = outv if isinstance(outv, pd.DataFrame) else pd.DataFrame()
    except Exception as e:
        print("LIANA villus-column failed:", e)
        df_v = pd.DataFrame()
else:
    print("Skip villus-column LIANA: too few cells in a group")

s_col = _lr_series_from_liana(df_v, "BEST4 Colonocytes", "BEST4 colonocyte")
s_lv = _lr_series_from_liana(df_v, "Lower Villus Enterocytes", "Villus bottom")
s_vt = _lr_series_from_liana(df_v, "Villus Tip Enterocytes", "Villus tip")
wide_vcol = pd.concat([s_col, s_lv, s_vt], axis=1)
if not wide_vcol.empty and wide_vcol.notna().any().any():
    plot_glia_epithelium_diff_heatmap(
        wide_vcol,
        axis_title="Glia ↔ column epithelium (BEST4 colon, lower villus, villus tip)",
        suptitle="Differential CCC (ileum): Glia vs villus-column enterocytes",
        outfile=FIG_DIR / "fig_epithelium_glia_villus_column_heatmap.pdf",
        exclude_cols_for_ranking=None,
        rank_by_column=None,
    )
else:
    print("Skip villus-column heatmap: empty or all-NaN wide table")

Ileum Glia+BEST4 cells for CFTR split: 2836
ad_cf var_names use gene symbols (post-mapping): n_var = 35574 ; CFTR present: True
BEST4 ileum: CFTR-hi 1027 CFTR-lo 1023 median log1p thr 1.6094
LIANA subset hi: {'BEST4_Ent_CFTRhi': 1027, 'Glia': 786}


/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:266: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationW

LIANA subset lo: {'BEST4_Ent_CFTRlo': 1023, 'Glia': 786}


/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:266: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationW

Note: rank_aggregate includes within-group (source==target) rows in df_hi/df_lo; heatmap Glia columns use only Glia↔epithelial pairs (_glia_epi_block), not autocrine.
--- df_hi: shape=(1167, 13)
  columns: ['source', 'target', 'ligand_complex', 'receptor_complex', 'lr_means', 'cellphone_pvals', 'expr_prod', 'scaled_weight', 'lr_logfc', 'spec_weight', 'lrscore', 'specificity_rank', 'magnitude_rank'] 
  has {'target', 'ligand_complex', 'source', 'receptor_complex'} ? {'target': True, 'ligand_complex': True, 'source': True, 'receptor_complex': True}
  sample Glia↔epi row (ligand_complex, receptor_complex): {'ligand_complex': 'S100A10', 'receptor_complex': 'CFTR'}
--- df_lo: shape=(1104, 13)
  columns: ['source', 'target', 'ligand_complex', 'receptor_complex', 'lr_means', 'cellphone_pvals', 'expr_prod', 'scaled_weight', 'lr_logfc', 'spec_weight', 'lrscore', 'specificity_rank', 'magnitude_rank'] 
  has {'target', 'ligand_complex', 'source', 'receptor_complex'} ? {'target': True, 'ligand_com

...

Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_diff_heatmap_best4_cftr_split.pdf
  De novo LIANA panel hits (Glia↔epi rows): df_hi = 50 df_lo = 41


maxp pruned
cmap pruned
1 extra bytes in post.stringData array
kern pruned
post pruned
Zapf dropped
feat dropped
meta dropped
morx dropped
'created' timestamp seems very low; regarding as unix timestamp
glyf pruned
Added gid0 to subset
Added first four glyphs to subset
Closing glyph list over 'glyf': 73 glyphs before
Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'X', 'a', 'ampersand', 'b', 'c', 'colon', 'comma', 'd', 'e', 'eight', 'emdash', 'endash', 'equal', 'f', 'five', 'four', 'g', 'h', 'hyphen', 'i', 'k', 'l', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'period', 'plus', 'r', 's', 'semicolon', 'seven', 'six', 'slash', 'space', 't', 'three', 'two', 'u', 'underscore', 'v', 'w', 'x', 'y', 'zero']
Glyph IDs:   [0, 1, 2, 3, 9, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 4

Neuropod/BEST4 ion/synaptic panel: L–R pairs matching curated genes: 309
Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_neuropod_epithelial_gene_panel.pdf
Ileum cells for villus-column LIANA: 74049


/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
/Users/kylekimler/miniforge3/envs/scanpy/lib/python3.11/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationW

Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig_epithelium_glia_villus_column_heatmap.pdf


In [11]:
df_hi

,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,specificity_rank,magnitude_rank
150,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,GUCA2A,GUCY2C,98.801437,0.000,582.499817,0.465978,inf,0.998622,0.985714,0.000155,0.000007
151,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,GUCA2B,GUCY2C,91.128555,0.000,536.567322,0.422275,inf,0.998719,0.985124,0.000355,0.000026
265,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,S100A10,CFTR,17.455683,0.000,271.931152,0.435845,155.135856,0.761938,0.979228,0.035359,0.000059
157,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,HLA-A,APLP2,10.070084,0.000,52.393082,0.341398,137.931851,0.606513,0.953902,0.093942,0.000237
824,Glia,BEST4_Ent_CFTRhi,S100A10,CFTR,9.485193,1.000,84.924347,0.008319,83.000321,0.237954,0.963430,0.642707,0.000421
...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,TGFB1,ITGB8,0.335443,0.759,0.083982,0.004592,-3.354572,0.266905,0.453095,1.000000,1.000000
292,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,TGFB1,ITGB5,0.302337,0.000,0.072957,0.171729,1.066342,0.560817,0.435724,0.605762,1.000000
259,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,PSEN1,NOTCH1,0.567671,0.000,0.150193,0.160562,4.054990,0.423553,0.525599,0.445058,1.000000
24,BEST4_Ent_CFTRhi,BEST4_Ent_CFTRhi,ANG,EGFR,0.311587,0.000,0.089910,0.171198,2.993143,0.630864,0.461560,0.245743,1.000000


## 4. Differential CCC (FARM ↔ CD4 subsets)
Compare **max lrscore per ligand–receptor** between CD4 populations (same ileum LIANA table). **Two heatmaps**: (1) all variable pairs; (2) **only** pairs where ligand/receptor tokens hit **OpenTargets GWAS** (`FARM_GWAS_GENES`) or **monocyte drug-target** (`FARM_DRUG_MONOCYTE_GENES`) lists — saved as `fig4_diff_ccc_farm_cd4_heatmap_opentargets.pdf`. **Tr1 vs Tfr** scatter and Δ plots use the same gene token logic — **not** neuro/axon colours (those are in §epithelium–Glia). NKT / γδ T omitted from CD4 heatmaps.

In [12]:
T_CD4 = [
    "CD4 Tr1",
    "CD4 Tfr",
    "CD4 Tfh",
    "CD4 pTreg",
    "CD4 Th17",
    "CD4 tTreg",
]

# OpenTargets / genetic vulnerability in FARMs (curated from your analysis)
FARM_GWAS_GENES = {
    "PTPN22",
    "FOSL2",
    "GPR35",
    "NDFIP1",
    "NR3C1",
    "TAGAP",
    "JAZF1",
    "IKZF1",
    "IRF5",
    "MAP3K8",
    "TSPAN14",
    "ITGB7",
    "LACC1",
    "PTPN2",
    "NCF4",
    "IFNGR2",
}
# Biologic & classical monocyte/macrophage drug targets (anti-TNF → TNFAIP3/TNF axis; vedolizumab integrins; ustekinumab IL-12/23; JAKi) — often lower on FARMs vs other monocyte subsets in druggability views
FARM_DRUG_MONOCYTE_GENES = {
    "TNFAIP3",
    "TNF",
    "TNFRSF1A",
    "TNFRSF1B",
    "IL12B",
    "IL23A",
    "IL23R",
    "IL12RB1",
    "IL12RB2",
    "JAK1",
    "JAK2",
    "TYK2",
    "ITGA4",
    "ITGB7",
    "NFKB1",
    "RELA",
}
GWAS_TOK = {g.upper() for g in FARM_GWAS_GENES}
DRUG_TOK = {g.upper() for g in FARM_DRUG_MONOCYTE_GENES}


def _lr_tokens(lig: str, rec: str) -> set[str]:
    return set(re.findall(r"[A-Za-z0-9]+", f"{lig}{rec}".upper()))


def farm_lr_is_opentarget_pair(lig: str, rec: str) -> bool:
    """True if ligand or receptor token matches GWAS or monocyte drug-target lists."""
    tok = _lr_tokens(lig, rec)
    return bool((tok & GWAS_TOK) | (tok & DRUG_TOK))


def farm_opentargets_category(row: pd.Series) -> str:
    lig = str(row.get("ligand", row.get("ligand_complex", "")))
    rec = str(row.get("receptor", row.get("receptor_complex", "")))
    tok = _lr_tokens(lig, rec)
    g = bool(tok & GWAS_TOK)
    d = bool(tok & DRUG_TOK)
    if g and d:
        return "both"
    if g:
        return "gwas"
    if d:
        return "drug"
    return "other"


series_list = []
for t in T_CD4:
    if t not in farm_tables or farm_tables[t].empty:
        continue
    m = max_lr_per_pair(farm_tables[t])
    s = m.set_index(["ligand_complex", "receptor_complex"])["lrscore"].rename(t)
    series_list.append(s)

if not series_list:
    raise RuntimeError("No CD4 FARM tables — re-run load with CD4 tTreg in watchlist.")

wide = pd.concat(series_list, axis=1)
# Pairs most variable across CD4 subsets (rank by std)
rank_var = wide.std(axis=1, skipna=True).dropna().sort_values(ascending=False)
if rank_var.empty:
    rank_var = wide.max(axis=1, skipna=True).sort_values(ascending=False)
top_n = min(40, len(rank_var))
pair_idx = rank_var.head(top_n).index
heat = wide.loc[pair_idx]

fig_h, ax_h = plt.subplots(figsize=(7.2, 8.8))
fig_h.subplots_adjust(bottom=0.2, top=0.86)
panel_label(ax_h, "a)")
data = np.ma.masked_invalid(heat.values.astype(float))
vmin_h = float(np.nanmin(heat.values))
vmax_h = float(np.nanmax(heat.values))
if vmin_h == vmax_h:
    vmax_h = vmin_h + 1e-9
im = ax_h.imshow(data, aspect="auto", cmap=NATURE_CMAP, vmin=vmin_h, vmax=vmax_h)
ax_h.set_yticks(np.arange(len(heat)))
ylabs = [shorten_label(f"{a} — {b}") for a, b in heat.index]
ax_h.set_yticklabels(ylabs, fontsize=5, color=NATURE_WONG["black"])
ax_h.set_xticks(np.arange(heat.shape[1]))
ax_h.set_xticklabels([c.replace("CD4 ", "") for c in heat.columns], rotation=45, ha="right", fontsize=6, color=NATURE_WONG["black"])
ax_h.set_title("FARM <-> CD4: variable ligand–receptor pairs (max lrscore)", fontsize=7, color=NATURE_WONG["black"])
cbar = fig_h.colorbar(im, ax=ax_h, fraction=0.035, pad=0.02)
cbar.ax.tick_params(colors=NATURE_WONG["black"])
cbar.set_label("lrscore (unitless)", color=NATURE_WONG["black"], fontsize=6)
for t in cbar.ax.get_yticklabels():
    t.set_color(NATURE_WONG["black"])
fig_h.suptitle("Differential CCC (ileum LIANA): FARM vs CD4 subsets", fontsize=7, color=NATURE_WONG["black"], y=0.98)
fig_h.text(
    0.5,
    0.06,
    LIANA_NOTE_FARM_HEATMAP,
    ha="center",
    va="top",
    fontsize=5.5,
    color=NATURE_WONG["black"],
    transform=fig_h.transFigure,
)
save_pdf(fig_h, FIG_DIR / "fig4_diff_ccc_farm_cd4_heatmap.pdf")

# --- FARM CD4 heatmap: OpenTargets GWAS + drug-target genes only ---
wide_ot = wide.loc[
    [farm_lr_is_opentarget_pair(str(a), str(b)) for a, b in wide.index],
    :,
]
if wide_ot.empty:
    print("No FARM CD4 pairs matched OpenTargets GWAS/drug lists (opentargets heatmap skipped).")
else:
    rank_var_ot = wide_ot.std(axis=1, skipna=True).dropna().sort_values(ascending=False)
    if rank_var_ot.empty:
        rank_var_ot = wide_ot.max(axis=1, skipna=True).sort_values(ascending=False)
    top_n_ot = min(40, len(rank_var_ot))
    heat_ot = wide_ot.loc[rank_var_ot.head(top_n_ot).index]
    fig_hot, ax_hot = plt.subplots(figsize=(7.2, 8.8))
    fig_hot.subplots_adjust(bottom=0.2, top=0.86)
    panel_label(ax_hot, "a)")
    data_ot = np.ma.masked_invalid(heat_ot.values.astype(float))
    vmin_ot = float(np.nanmin(heat_ot.values))
    vmax_ot = float(np.nanmax(heat_ot.values))
    if vmin_ot == vmax_ot:
        vmax_ot = vmin_ot + 1e-9
    im_ot = ax_hot.imshow(data_ot, aspect="auto", cmap=NATURE_CMAP, vmin=vmin_ot, vmax=vmax_ot)
    ax_hot.set_yticks(np.arange(len(heat_ot)))
    ax_hot.set_yticklabels(
        [shorten_label(f"{a} — {b}") for a, b in heat_ot.index],
        fontsize=5,
        color=NATURE_WONG["black"],
    )
    ax_hot.set_xticks(np.arange(heat_ot.shape[1]))
    ax_hot.set_xticklabels(
        [c.replace("CD4 ", "") for c in heat_ot.columns],
        rotation=45,
        ha="right",
        fontsize=6,
        color=NATURE_WONG["black"],
    )
    ax_hot.set_title(
        "FARM <-> CD4: variable pairs (GWAS / drug-target genes only)",
        fontsize=7,
        color=NATURE_WONG["black"],
    )
    cbar_ot = fig_hot.colorbar(im_ot, ax=ax_hot, fraction=0.035, pad=0.02)
    cbar_ot.ax.tick_params(colors=NATURE_WONG["black"])
    cbar_ot.set_label("lrscore (unitless)", color=NATURE_WONG["black"], fontsize=6)
    for t in cbar_ot.ax.get_yticklabels():
        t.set_color(NATURE_WONG["black"])
    fig_hot.suptitle(
        "Differential CCC (ileum LIANA): FARM vs CD4 (OpenTargets GWAS & drug targets)",
        fontsize=7,
        color=NATURE_WONG["black"],
        y=0.98,
    )
    fig_hot.text(
        0.5,
        0.06,
        LIANA_NOTE_FARM_HEATMAP,
        ha="center",
        va="top",
        fontsize=5.5,
        color=NATURE_WONG["black"],
        transform=fig_hot.transFigure,
    )
    save_pdf(fig_hot, FIG_DIR / "fig4_diff_ccc_farm_cd4_heatmap_opentargets.pdf")

# --- Tr1 vs Tfr: scatter of matched pairs ---
tr1 = max_lr_per_pair(farm_tables["CD4 Tr1"]).rename(columns={"lrscore": "CD4 Tr1"})
tfr = max_lr_per_pair(farm_tables["CD4 Tfr"]).rename(columns={"lrscore": "CD4 Tfr"})
m12 = tr1.merge(tfr, on=["ligand_complex", "receptor_complex"], how="inner")
m12 = m12.rename(columns={"ligand_complex": "ligand", "receptor_complex": "receptor"})
m12["farm_cat"] = m12.apply(farm_opentargets_category, axis=1)

fig_s, ax_s = plt.subplots(figsize=(3.5, 3.5))
panel_label(ax_s, "b)")
farm_style = {
    "other": (NATURE_WONG["yellow"], 8, 0.45, "Other L–R pairs"),
    "gwas": (NATURE_WONG["sky_blue"], 11, 0.8, "GWAS vulnerability (FARM)"),
    "drug": (NATURE_WONG["bluish_green"], 11, 0.8, "Biologic / monocyte drug targets"),
    "both": (NATURE_WONG["vermillion"], 14, 0.92, "GWAS + drug-target genes"),
}
for cat in ['other', 'gwas', 'drug', 'both']:
    m = m12['farm_cat'] == cat
    if not m.any():
        continue
    ccol, siz, alp, lab = farm_style[cat]
    ax_s.scatter(
        m12.loc[m, 'CD4 Tr1'],
        m12.loc[m, 'CD4 Tfr'],
        s=siz,
        c=ccol,
        edgecolors=NATURE_WONG['black'],
        linewidths=0.35,
        alpha=alp,
        label=lab,
    )
lim = float(max(m12["CD4 Tr1"].max(), m12["CD4 Tfr"].max()))
ax_s.plot([0, lim], [0, lim], color=NATURE_WONG["black"], linewidth=0.6, linestyle="--", alpha=0.7)
ax_s.set_xlabel("lrscore FARM <-> CD4 Tr1 (unitless)", color=NATURE_WONG["black"])
ax_s.set_ylabel("lrscore FARM <-> CD4 Tfr (unitless)", color=NATURE_WONG["black"])
ax_s.set_title("Matched pairs: Tr1 vs Tfr", fontsize=7, color=NATURE_WONG["black"])
leg = ax_s.legend(frameon=False, loc="upper left", handlelength=1.2, fontsize=5.5)
for text in leg.get_texts():
    text.set_color(NATURE_WONG["black"])
ax_s.tick_params(colors=NATURE_WONG["black"])
fig_s.subplots_adjust(bottom=0.2)
fig_s.suptitle("Differential CCC: Tr1 vs Tfr (OpenTargets GWAS & drug targets)", fontsize=7, color=NATURE_WONG["black"], y=1.08)
fig_s.text(
    0.5,
    0.03,
    LIANA_NOTE_SHORT,
    ha="center",
    va="bottom",
    fontsize=5,
    color=NATURE_WONG["black"],
    transform=fig_s.transFigure,
)
save_pdf(fig_s, FIG_DIR / "fig5_tr1_vs_tfr_scatter.pdf")

# --- Bar: largest Tr1 − Tfr differences ---
m12["delta_tr1_minus_tfr"] = m12["CD4 Tr1"] - m12["CD4 Tfr"]
topd = m12.reindex(m12["delta_tr1_minus_tfr"].abs().sort_values(ascending=False).index).head(18)
fig_d, ax_d = plt.subplots(figsize=(3.5, 5.5))
panel_label(ax_d, "c)")
y = np.arange(len(topd))
col_farm = {
    "other": NATURE_WONG["yellow"],
    "gwas": NATURE_WONG["sky_blue"],
    "drug": NATURE_WONG["bluish_green"],
    "both": NATURE_WONG["vermillion"],
}
cols_bar = [col_farm[farm_opentargets_category(row)] for _, row in topd.iterrows()]
ax_d.barh(y, topd["delta_tr1_minus_tfr"], color=cols_bar, edgecolor=NATURE_WONG["black"], linewidth=0.35)
labs = [shorten_label(f"{a} — {b}") for a, b in zip(topd["ligand"], topd["receptor"])]
ax_d.set_yticks(y)
ax_d.set_yticklabels(labs, fontsize=5.5, color=NATURE_WONG["black"])
ax_d.invert_yaxis()
ax_d.set_xlabel("Δ lrscore (Tr1 − Tfr) (unitless)", color=NATURE_WONG["black"])
ax_d.set_title("Largest |Tr1−Tfr| shifts (FARM-linked)", fontsize=7, color=NATURE_WONG["black"])
ax_d.axvline(0, color=NATURE_WONG["black"], linewidth=0.5)
ax_d.tick_params(colors=NATURE_WONG["black"])
ax_d.spines["top"].set_visible(False)
ax_d.spines["right"].set_visible(False)
leg_p = [
    Patch(facecolor=NATURE_WONG["vermillion"], edgecolor=NATURE_WONG["black"], label="GWAS + drug-target"),
    Patch(facecolor=NATURE_WONG["sky_blue"], edgecolor=NATURE_WONG["black"], label="GWAS (FARM)"),
    Patch(facecolor=NATURE_WONG["bluish_green"], edgecolor=NATURE_WONG["black"], label="Drug targets"),
    Patch(facecolor=NATURE_WONG["yellow"], edgecolor=NATURE_WONG["black"], label="Other"),
]
leg2 = ax_d.legend(handles=leg_p, frameon=False, loc="lower right", fontsize=5.5)
for t in leg2.get_texts():
    t.set_color(NATURE_WONG["black"])
fig_d.subplots_adjust(bottom=0.18)
fig_d.suptitle("Differential CCC: Tr1 vs Tfr (OpenTargets)", fontsize=7, color=NATURE_WONG["black"], y=1.02)
fig_d.text(
    0.5,
    0.02,
    LIANA_NOTE_SHORT,
    ha="center",
    va="bottom",
    fontsize=5,
    color=NATURE_WONG["black"],
    transform=fig_d.transFigure,
)
save_pdf(fig_d, FIG_DIR / "fig6_delta_tr1_minus_tfr.pdf")

maxp pruned
cmap pruned
1 extra bytes in post.stringData array
kern pruned
post pruned
Zapf dropped
feat dropped
meta dropped
morx dropped
'created' timestamp seems very low; regarding as unix timestamp
glyf pruned
Added gid0 to subset
Added first four glyphs to subset
Closing glyph list over 'glyf': 68 glyphs before
Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'V', 'X', 'Z', 'a', 'b', 'c', 'colon', 'comma', 'd', 'e', 'eight', 'emdash', 'endash', 'equal', 'f', 'five', 'four', 'g', 'greater', 'h', 'hyphen', 'i', 'l', 'less', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'period', 'r', 's', 'semicolon', 'seven', 'six', 'space', 't', 'three', 'two', 'u', 'underscore', 'v', 'x', 'zero']
Glyph IDs:   [0, 1, 2, 3, 11, 12, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 53, 54, 55, 57, 59, 61, 66,

Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig4_diff_ccc_farm_cd4_heatmap.pdf
Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig4_diff_ccc_farm_cd4_heatmap_opentargets.pdf


maxp pruned
cmap pruned
1 extra bytes in post.stringData array
kern pruned
post pruned
Zapf dropped
feat dropped
meta dropped
morx dropped
'created' timestamp seems very low; regarding as unix timestamp
glyf pruned
Added gid0 to subset
Added first four glyphs to subset
Closing glyph list over 'glyf': 56 glyphs before
Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'F', 'G', 'L', 'M', 'O', 'R', 'S', 'T', 'W', 'a', 'ampersand', 'b', 'c', 'colon', 'comma', 'd', 'e', 'eight', 'endash', 'equal', 'f', 'four', 'g', 'greater', 'h', 'hyphen', 'i', 'l', 'less', 'm', 'n', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'period', 'plus', 'r', 's', 'semicolon', 'six', 'space', 't', 'two', 'u', 'v', 'x', 'zero']
Glyph IDs:   [0, 1, 2, 3, 9, 11, 12, 14, 15, 16, 17, 19, 20, 21, 23, 25, 27, 29, 30, 31, 32, 33, 36, 37, 38, 39, 41, 42, 47, 48, 50, 53, 54, 55, 58, 68, 69, 70, 71, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 85, 86, 87, 88, 89, 91, 178]
Closed glyph list over 'glyf': 56 gl

Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig5_tr1_vs_tfr_scatter.pdf
Saved /Users/kylekimler/Projects/GCA/github_vignette_output/LIANA/ileum_figures/fig6_delta_tr1_minus_tfr.pdf


## Follow-up
- **FARM scatter / Δ**: refine `FARM_GWAS_GENES` and `FARM_DRUG_MONOCYTE_GENES` against your OpenTargets exports; token matching is exact symbol overlap in LIANA `ligand_complex` / `receptor_complex` strings.
- **Glia scatter / Δ**: default contrast is **EEC (mature)** vs **Tuft** (falls back to the first two columns if missing); change `SC_A` / `SC_B` in the notebook. **Heatmaps** exclude **Villus bottom** from **row ranking** only (column still shown); edit `GLIA_HEATMAP_CTRL_COL` / `reduce_canonical_neuropeptide_outgoing` if needed.
- **BEST4 CFTR split** (all / neuro-axon differential heatmaps, focused CFTR heatmap, gene-panel, villus-column): `fig_epithelium_glia_diff_heatmap_all_cftr_split.pdf`, `fig_epithelium_glia_diff_heatmap_neuro_axon_cftr_split.pdf`, `fig_epithelium_glia_diff_heatmap_best4_cftr_split.pdf`, `fig_epithelium_glia_neuropod_epithelial_gene_panel.pdf`, `fig_epithelium_glia_villus_column_heatmap.pdf` — require **`liana`**; run the **“Epithelium–Glia: BEST4 CFTR split & villus columns”** cell **after** `wide_all` / `wide_na` are built; uses median **log1p CFTR** on ileal BEST4 enterocytes.
- Refine neuro / axon keyword lists (epithelium–Glia) against curated databases if needed.
- Optional: stricter CSV pre-filter to shrink `dfw` if memory is tight.
- **Vector workflow**: edit PDFs in Illustrator if needed, then save **.ai** for production; otherwise submit **.pdf** from above.
- **Neuropod / NCAM / GDNF–RET (§2.2):** `neuropod_glia_eec_focus_summary.csv`, `fig_neuropod_glia_*.pdf` (includes `*_GDNF_RET_GFRA*.pdf`), `neurofil_ncam_mean_log1p_by_celltype_region.csv`, `fig_neuropod_neurofil_ncam_expr_ileum_bars.pdf`.
